# B마트 V2 수요예측·재고정책 보완본

이 노트북은 **매일 전날 실적까지 확보한 뒤 다음 날을 예측하는 1일 선행 모델**입니다.

- `Demand`는 데이터셋이 제공한 수요 지표이며 실제 잠재수요로 단정하지 않습니다.
- 정적 정책 비교는 리드타임 0일 가정입니다.
- 동적 시뮬레이션은 누적 리드타임 수요 Quantile을 직접 예측해 사용하지만 비용·MOQ·공급제약이 없는 검증용 시뮬레이션입니다.
- Test 기간은 모델·가설 판단용 EDA에서 제외합니다.

- 누적수요 모델은 발주 시점에 확정된 향후 5일 가격·할인·프로모션·휴일 계획을 집계 feature로 사용합니다. 계획이 미확정이면 별도 시나리오 입력이 필요합니다.
- 발주량은 현재고+입고예정-미납으로 계산한 재고 포지션을 기준으로 하며, MOQ와 박스 단위를 적용합니다. 실제 원가 미입력 시 비용 비교는 비활성화됩니다.
- `Demand`는 데이터셋 제공 라벨이며 잠재수요·lost sales 정답으로 검증되지 않았습니다. 따라서 서비스 수준과 재고비용은 운영 확정치가 아니라 시나리오 비교로 해석합니다.
- 배포에서는 노트북 내부 상태 대신 독립 추론 모듈이 config와 모델 5개를 로드합니다. artifact manifest의 SHA256으로 배포 파일 손상을 검증합니다.
- 운영 중에는 주 단위 WAPE·Bias·P90/P95 coverage·crossing·PSI·결측률을 추적합니다. 입력 장애는 즉시 중단하고, 반복 경고는 재보정 후 재학습 순으로 처리합니다.
- The 5-day cumulative P90/P95 forecasts use a validation-selected, maturity-aware rolling conformal correction. The fixed correction remains as an explicit fallback, and stale calibration is surfaced at inference.


In [ ]:
# ============================================================
# [환경 설정] 재현 가능한 공통 설정
# ============================================================

%pip install -q catboost koreanize-matplotlib

import hashlib
import json
import os
import platform
import warnings
from pathlib import Path

import catboost
import koreanize_matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn

from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_pinball_loss,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("default")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 현재 검증 계약: 매일 전날 실적까지 확보한 뒤 다음 날을 예측한다.
FORECAST_HORIZON_DAYS = 1

DEV_MODEL_PARAMS = {
    "iterations": 1200,
    "learning_rate": 0.05,
    "depth": 7,
    "loss_function": "RMSE",
    "eval_metric": "MAE",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
}

FINAL_POINT_PARAMS = {
    "iterations": 2000,
    "learning_rate": 0.05,
    "depth": 7,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
}

FINAL_QUANTILE_PARAMS = {
    "iterations": 2000,
    "learning_rate": 0.05,
    "depth": 7,
    "random_seed": RANDOM_STATE,
    "allow_writing_files": False,
}

print("환경 설정 완료")
print("Random Seed:", RANDOM_STATE)
print("Forecast Horizon:", FORECAST_HORIZON_DAYS, "day")
print("Python:", platform.python_version())
print("CatBoost:", catboost.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("scipy:", scipy.__version__)

In [ ]:
# ============================================================
# [데이터 연결] Google Drive를 먼저 마운트
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Colab이 아닌 환경입니다. BMART_DATA_PATH 환경변수를 사용합니다.")

DEFAULT_DATA_PATH = "/content/drive/MyDrive/MBCA/Machine Learning/project/B_mart.csv"
DATA_PATH = Path(os.environ.get("BMART_DATA_PATH", DEFAULT_DATA_PATH))

print("데이터 경로:", DATA_PATH)

In [ ]:
# ============================================================
# [데이터 로드] 원본 파일과 hash 기록
# ============================================================

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {DATA_PATH}\n"
        "Colab Drive 경로 또는 BMART_DATA_PATH를 확인하세요."
    )

DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
Bmart = pd.read_csv(DATA_PATH)

print("===== 실제 컬럼 목록 =====")
for i, col in enumerate(Bmart.columns, start=1):
    print(i, repr(col))

print("\n행 수:", len(Bmart))
print("SHA256:", DATA_SHA256)
display(Bmart.head())

In [ ]:
# ============================================================
# [전처리] 컬럼명 표준화 + 기본 데이터 검증
# ============================================================

Bmart = Bmart.rename(columns={
    "Date": "date",
    "Day_of_Week": "day_of_week",
    "Is_Holiday": "is_holiday",
    "Store ID": "store_id",
    "Product ID": "product_id",
    "Category": "category",
    "Region": "region",
    "Inventory Level": "inventory_level",
    "Units Sold": "units_sold",
    "Units Ordered": "units_ordered",
    "Price": "price",
    "Discount": "discount_pct",
    "Weather Condition": "weather_condition",
    "Promotion": "promotion_flag",
    "Competitor Pricing": "competitor_pricing",
    "Seasonality": "seasonality",
    "Epidemic": "epidemic",
    "Demand": "demand_qty",
})

required_columns = {
    "date", "day_of_week", "is_holiday", "store_id", "product_id",
    "category", "region", "inventory_level", "units_sold",
    "units_ordered", "price", "discount_pct", "weather_condition",
    "promotion_flag", "competitor_pricing", "seasonality", "epidemic",
    "demand_qty",
}

missing_columns = sorted(required_columns - set(Bmart.columns))
if missing_columns:
    raise ValueError(f"필수 컬럼 누락: {missing_columns}")

Bmart["date"] = pd.to_datetime(Bmart["date"], errors="raise")
Bmart = Bmart.sort_values(
    ["date", "store_id", "product_id"]
).reset_index(drop=True)

key_columns = ["date", "store_id", "product_id"]
duplicate_count = Bmart.duplicated(key_columns).sum()
if duplicate_count:
    raise ValueError(f"date-store-product 중복 {duplicate_count}건 발견")

group_date_diff = (
    Bmart.groupby(["store_id", "product_id"])["date"]
    .diff()
    .dropna()
)
non_daily_gap_count = (group_date_diff != pd.Timedelta(days=1)).sum()

category_instability = (
    Bmart.groupby(["store_id", "product_id"])["category"].nunique().gt(1).sum()
)
region_instability = Bmart.groupby("store_id")["region"].nunique().gt(1).sum()

print("===== 데이터 검증 =====")
print("키 중복:", duplicate_count)
print("일 단위 비연속 구간:", non_daily_gap_count)
print("매장×상품 category 변경 조합:", category_instability)
print("매장별 region 변경:", region_instability)

if non_daily_gap_count:
    print("주의: shift(7)은 7일 전이 아니라 7개 관측치 전이 될 수 있습니다.")
if category_instability:
    print("주의: category가 기준정보라면 변경 원인을 확인해야 합니다.")

## 1. Target·데이터 품질 검증

In [ ]:
# ============================================================
# [Target 진단] Demand는 데이터셋 제공 수요 지표로만 해석
# ============================================================

check_columns = [
    "demand_qty", "units_sold", "inventory_level", "units_ordered",
    "price", "discount_pct", "competitor_pricing", "promotion_flag",
]

corr_matrix = Bmart[check_columns].corr(numeric_only=True)
display(
    corr_matrix[["demand_qty"]]
    .sort_values("demand_qty", ascending=False)
    .round(3)
)

Bmart["demand_sales_gap"] = Bmart["demand_qty"] - Bmart["units_sold"]
Bmart["sold_out_candidate"] = (
    (Bmart["inventory_level"] > 0)
    & (Bmart["units_sold"] >= Bmart["inventory_level"])
)

print("Demand > Units Sold:", (Bmart["demand_sales_gap"] > 0).sum())
print("Demand = Units Sold:", (Bmart["demand_sales_gap"] == 0).sum())
print("Demand < Units Sold:", (Bmart["demand_sales_gap"] < 0).sum())
print(
    "주의: Demand < Units Sold가 존재하므로 Demand를 실제 잠재수요나 "
    "lost sales의 정답으로 단정하지 않습니다."
)

stock_compare = (
    Bmart.groupby("sold_out_candidate")
    .agg(
        count=("demand_qty", "size"),
        avg_inventory=("inventory_level", "mean"),
        avg_units_sold=("units_sold", "mean"),
        avg_demand=("demand_qty", "mean"),
        avg_units_ordered=("units_ordered", "mean"),
        avg_gap=("demand_sales_gap", "mean"),
    )
    .round(2)
)
display(stock_compare)


# ------------------------------------------------------------
# Target 의미 계약: 성능 수치보다 먼저 해석 범위를 고정한다.
# ------------------------------------------------------------
TARGET_CONTRACT = {
    "column": "demand_qty",
    "source_column": "Demand",
    "status": "warning_unverified_semantics",
    "operational_interpretation": (
        "데이터셋이 제공한 수요 유사 라벨. 잠재수요·실제 주문요청량으로 검증되지 않음."
    ),
    "approved_uses": [
        "동일 라벨 기준의 시계열 예측 성능 비교",
        "가정이 명시된 재고정책 시나리오 비교",
    ],
    "prohibited_claims": [
        "latent demand 또는 lost sales 정답으로 단정",
        "실제 품절·서비스 수준의 확정적 운영 KPI로 단정",
        "프로모션·할인의 인과효과로 해석",
    ],
    "unresolved_fields": [
        "Demand 생성 규칙",
        "Inventory Level의 측정 시점",
        "Units Ordered의 발주일·입고일 연결",
        "결품 시 미충족 수요 기록 방식",
    ],
}

target_audit_df = pd.DataFrame([
    {
        "Check": "Demand < Units Sold",
        "Rows": int((Bmart["demand_qty"] < Bmart["units_sold"]).sum()),
        "Ratio_Pct": float(
            (Bmart["demand_qty"] < Bmart["units_sold"]).mean() * 100
        ),
        "Interpretation": "Demand를 실제 총수요로 단정하기 어려운 불일치",
    },
    {
        "Check": "Demand = Units Sold",
        "Rows": int((Bmart["demand_qty"] == Bmart["units_sold"]).sum()),
        "Ratio_Pct": float(
            (Bmart["demand_qty"] == Bmart["units_sold"]).mean() * 100
        ),
        "Interpretation": "판매량과 동일한 관측",
    },
    {
        "Check": "Demand > Units Sold",
        "Rows": int((Bmart["demand_qty"] > Bmart["units_sold"]).sum()),
        "Ratio_Pct": float(
            (Bmart["demand_qty"] > Bmart["units_sold"]).mean() * 100
        ),
        "Interpretation": "미충족수요 가능성은 있으나 생성 규칙 미확인",
    },
])

print("===== Target 의미 계약 =====")
print("상태:", TARGET_CONTRACT["status"])
print("해석:", TARGET_CONTRACT["operational_interpretation"])
display(target_audit_df.round(2))

In [ ]:
# ============================================================
# [Feature 계약] 1일 선행 예측 기준
# ============================================================

target = "demand_qty"

leakage_columns = ["units_sold", "units_ordered", "inventory_level"]

# 실제 운영에서는 아래 값이 예측일 전에 제공되어야 한다.
planned_or_forecast_features = [
    "price", "discount_pct", "promotion_flag",
    "weather_condition", "competitor_pricing", "epidemic",
]

base_features = [
    "day_of_week", "is_holiday", "store_id", "product_id", "category",
    "region", "price", "discount_pct", "weather_condition",
    "promotion_flag", "competitor_pricing", "seasonality", "epidemic",
]

assert target not in base_features
assert not (set(leakage_columns) & set(base_features))

print("Target:", target)
print("제외 변수:", leakage_columns)
print("사전 확보 필요 변수:", planned_or_forecast_features)
print("기본 Feature 수:", len(base_features))

## 2. Feature 생성과 시계열 분할

In [ ]:
# ============================================================
# [V2 셀 6] Demand 기반 시계열 Feature 생성
# ============================================================

# ------------------------------------------------------------
# 1. 매장-상품 단위 그룹
# ------------------------------------------------------------

group_cols = [
    "store_id",
    "product_id"
]


# ------------------------------------------------------------
# 2. Lag Feature
# 과거 특정 시점의 수요
# ------------------------------------------------------------

for lag in [1, 7, 14, 28]:

    Bmart[f"demand_lag_{lag}"] = (
        Bmart
        .groupby(group_cols)["demand_qty"]
        .shift(lag)
    )


# ------------------------------------------------------------
# 3. Rolling Mean
#
# shift(1)을 먼저 적용
# → 오늘 수요는 제외하고
#   어제까지의 데이터만 사용
# ------------------------------------------------------------

Bmart["demand_rolling_mean_7"] = (
    Bmart
    .groupby(group_cols)["demand_qty"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .mean()
    )
)


Bmart["demand_rolling_mean_28"] = (
    Bmart
    .groupby(group_cols)["demand_qty"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=28,
            min_periods=28
        )
        .mean()
    )
)


# ------------------------------------------------------------
# 4. Rolling Standard Deviation
# 최근 수요 변동성
# ------------------------------------------------------------

Bmart["demand_rolling_std_7"] = (
    Bmart
    .groupby(group_cols)["demand_qty"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .std()
    )
)


# ------------------------------------------------------------
# 5. 생성 확인
# ------------------------------------------------------------

time_features = [
    "demand_lag_1",
    "demand_lag_7",
    "demand_lag_14",
    "demand_lag_28",
    "demand_rolling_mean_7",
    "demand_rolling_mean_28",
    "demand_rolling_std_7"
]


print("===== 생성된 시계열 Feature =====")

for col in time_features:
    print(
        col,
        "- 결측치:",
        Bmart[col].isna().sum()
    )


print("\n===== 예시 =====")

display(
    Bmart[
        [
            "date",
            "store_id",
            "product_id",
            "demand_qty"
        ]
        +
        time_features
    ]
    .head(35)
)


In [ ]:
# ============================================================
# [V2 셀 7] 날짜 Feature 생성 + 모델링 데이터 정리
# ============================================================

# ------------------------------------------------------------
# 1. 날짜 Feature 생성
# ------------------------------------------------------------

Bmart["year"] = Bmart["date"].dt.year
Bmart["month"] = Bmart["date"].dt.month
Bmart["day"] = Bmart["date"].dt.day

# 월의 순환성을 조금 더 잘 표현하기 위한 Feature
Bmart["month_sin"] = np.sin(
    2 * np.pi * Bmart["month"] / 12
)

Bmart["month_cos"] = np.cos(
    2 * np.pi * Bmart["month"] / 12
)


# ------------------------------------------------------------
# 2. 최종 Feature 후보
# ------------------------------------------------------------

feature_columns = (
    base_features
    +
    [
        "year",
        "month",
        "day",
        "month_sin",
        "month_cos"
    ]
    +
    time_features
)


print("===== 최종 Feature 후보 =====")

for col in feature_columns:
    print("-", col)

print("\nFeature 개수:", len(feature_columns))


# ------------------------------------------------------------
# 3. 모델링용 데이터 생성
#
# lag / rolling 결측치가 있는
# 초기 28일 제거
# ------------------------------------------------------------

model_df = Bmart.dropna(
    subset=time_features
).copy()


# ------------------------------------------------------------
# 4. 결과 확인
# ------------------------------------------------------------

print("\n===== 모델링 데이터 =====")

print(
    "원본 행 수:",
    len(Bmart)
)

print(
    "모델링 행 수:",
    len(model_df)
)

print(
    "제거된 행 수:",
    len(Bmart) - len(model_df)
)

print(
    "시작일:",
    model_df["date"].min()
)

print(
    "종료일:",
    model_df["date"].max()
)

print(
    "날짜 수:",
    model_df["date"].nunique()
)


# ------------------------------------------------------------
# 5. 결측치 최종 확인
# ------------------------------------------------------------

print("\n===== Feature 결측치 =====")

print(
    model_df[
        feature_columns
    ]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
)

In [ ]:
# ============================================================
# [시계열 분할] Train / Validation / Test
# ============================================================

train_end = pd.Timestamp("2023-07-31")
valid_end = pd.Timestamp("2023-10-31")

train_df = model_df[model_df["date"] <= train_end].copy()
valid_df = model_df[
    (model_df["date"] > train_end) & (model_df["date"] <= valid_end)
].copy()
test_df = model_df[model_df["date"] > valid_end].copy()

assert train_df["date"].max() < valid_df["date"].min()
assert valid_df["date"].max() < test_df["date"].min()
assert len(train_df) + len(valid_df) + len(test_df) == len(model_df)

# 모델/가설 판단용 EDA에서 Test target을 보지 않도록 개발 구간만 사용한다.
eda_df = pd.concat([train_df, valid_df], ignore_index=True)

for name, frame in [
    ("TRAIN", train_df), ("VALIDATION", valid_df), ("TEST", test_df)
]:
    print(
        name,
        frame["date"].min(), "~", frame["date"].max(),
        "rows=", len(frame),
    )

print("EDA rows (Test 제외):", len(eda_df))

## 3. Point Forecast 학습과 검증

In [ ]:
# ============================================================
# [V2 셀 9] Baseline 모델 평가
# 최근 7일 평균 수요를 예측값으로 사용
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ------------------------------------------------------------
# 평가 함수
# ------------------------------------------------------------

def evaluate_regression(
    y_true,
    y_pred,
    model_name="Model"
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    wape = (
        np.sum(
            np.abs(
                y_true - y_pred
            )
        )
        /
        np.sum(
            np.abs(y_true)
        )
        * 100
    )

    bias = np.mean(
        y_pred - y_true
    )


    print(
        f"===== {model_name} ====="
    )

    print(
        f"MAE  : {mae:.4f}"
    )

    print(
        f"RMSE : {rmse:.4f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )

    print(
        f"WAPE : {wape:.4f}%"
    )

    print(
        f"Bias : {bias:.4f}"
    )


    return {

        "Model":
            model_name,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "WAPE":
            wape,

        "Bias":
            bias
    }


# ------------------------------------------------------------
# Baseline 예측
# ------------------------------------------------------------

baseline_pred = (
    valid_df[
        "demand_rolling_mean_7"
    ].values
)


# ------------------------------------------------------------
# Validation 평가
# ------------------------------------------------------------

baseline_result = evaluate_regression(
    valid_df["demand_qty"].values,
    baseline_pred,
    model_name="V2 7-day Rolling Mean Baseline"
)

In [ ]:
# ============================================================
# [V2 셀 10] 기본 CatBoost 모델
# ============================================================

# ------------------------------------------------------------
# 1. 범주형 Feature
# ------------------------------------------------------------

categorical_features = [
    "day_of_week",
    "store_id",
    "product_id",
    "category",
    "region",
    "weather_condition",
    "seasonality"
]


# ------------------------------------------------------------
# 2. Train / Validation
# ------------------------------------------------------------

X_train = train_df[
    feature_columns
].copy()

y_train = train_df[
    target
].copy()


X_valid = valid_df[
    feature_columns
].copy()

y_valid = valid_df[
    target
].copy()


# ------------------------------------------------------------
# 3. 범주형 Feature 위치
# ------------------------------------------------------------

cat_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]


# ------------------------------------------------------------
# 4. 모델 생성
# ------------------------------------------------------------

v2_catboost_model = CatBoostRegressor(
    iterations=1200,
    learning_rate=0.05,
    depth=7,

    loss_function="RMSE",
    eval_metric="MAE",

    random_seed=RANDOM_STATE,

    verbose=100,
    allow_writing_files=False
)


# ------------------------------------------------------------
# 5. 학습
# ------------------------------------------------------------

v2_catboost_model.fit(
    X_train,
    y_train,

    cat_features=cat_indices,

    eval_set=(
        X_valid,
        y_valid
    ),

    early_stopping_rounds=100,

    verbose=100
)


# ------------------------------------------------------------
# 6. Validation 예측
# ------------------------------------------------------------

v2_valid_pred = (
    v2_catboost_model.predict(
        X_valid
    )
)

v2_valid_pred = np.maximum(
    v2_valid_pred,
    0
)


# ------------------------------------------------------------
# 7. 평가
# ------------------------------------------------------------

v2_catboost_result = evaluate_regression(
    y_valid.values,
    v2_valid_pred,
    model_name="V2 CatBoost"
)


print(
    "\nBest iteration:",
    v2_catboost_model.get_best_iteration()
)

In [ ]:
# ============================================================
# [V2 셀 11] CatBoost Feature Importance 확인
# ============================================================

feature_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": v2_catboost_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print("===== Feature Importance =====")

display(
    feature_importance
    .round(3)
)


# ------------------------------------------------------------
# 상위 15개 Feature 시각화
# ------------------------------------------------------------

plt.figure(figsize=(10, 7))

top15 = feature_importance.head(15)

plt.barh(
    top15["feature"],
    top15["importance"]
)

plt.gca().invert_yaxis()

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("V2 CatBoost Feature Importance")

plt.grid(
    axis="x",
    alpha=0.3
)

plt.show()

In [ ]:
# ============================================================
# [V2 셀 12] Price 구조 검사
# ============================================================

# ------------------------------------------------------------
# 1. 상품별 가격 종류 수
# ------------------------------------------------------------

price_by_product = (
    eda_df
    .groupby("product_id")
    .agg(
        price_nunique=("price", "nunique"),
        price_mean=("price", "mean"),
        price_std=("price", "std"),
        demand_mean=("demand_qty", "mean")
    )
    .round(3)
)


print("===== 상품별 Price 구조 =====")

display(
    price_by_product
)


# ------------------------------------------------------------
# 2. 상품별 Category 개수
# ------------------------------------------------------------

category_by_product = (
    eda_df
    .groupby("product_id")["category"]
    .nunique()
)


print("\n===== 상품별 Category 개수 =====")

print(
    category_by_product.value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 3. Category별 평균 가격 / 평균 Demand
# ------------------------------------------------------------

category_summary = (
    eda_df
    .groupby("category")
    .agg(
        count=("demand_qty", "size"),
        avg_price=("price", "mean"),
        avg_demand=("demand_qty", "mean"),
        avg_discount=("discount_pct", "mean"),
        promotion_rate=("promotion_flag", "mean")
    )
    .round(2)
    .sort_values(
        "avg_demand",
        ascending=False
    )
)


print("\n===== Category별 비교 =====")

display(
    category_summary
)


# ------------------------------------------------------------
# 4. 전체 가격 종류
# ------------------------------------------------------------

print("\n===== Price 기본 정보 =====")

print(
    "전체 가격 종류:",
    eda_df["price"].nunique()
)

print(
    "상품 수:",
    eda_df["product_id"].nunique()
)

print(
    "가격 평균:",
    round(eda_df["price"].mean(), 2)
)

print(
    "가격 표준편차:",
    round(eda_df["price"].std(), 2)
)

In [ ]:
# ============================================================
# [V2 셀 14] Price와 Demand 관계 점검
# ============================================================

# ------------------------------------------------------------
# 1. 전체 상관관계
# ------------------------------------------------------------

price_corr = eda_df[
    ["price", "demand_qty"]
].corr().iloc[0, 1]

print("===== Price ↔ Demand 상관계수 =====")
print(round(price_corr, 4))


# ------------------------------------------------------------
# 2. 가격 구간별 평균 Demand
# ------------------------------------------------------------

eda_df["price_bin"] = pd.qcut(
    eda_df["price"],
    q=10,
    duplicates="drop"
)

price_bin_summary = (
    eda_df
    .groupby("price_bin", observed=True)
    .agg(
        count=("demand_qty", "size"),
        avg_price=("price", "mean"),
        avg_demand=("demand_qty", "mean"),
        avg_discount=("discount_pct", "mean"),
        promotion_rate=("promotion_flag", "mean")
    )
    .round(2)
)

print("\n===== 가격 구간별 평균 Demand =====")
display(price_bin_summary)


# ------------------------------------------------------------
# 3. Promotion 여부별 가격-Demand 관계
# ------------------------------------------------------------

promotion_summary = (
    eda_df
    .groupby("promotion_flag")
    .agg(
        count=("demand_qty", "size"),
        avg_price=("price", "mean"),
        avg_discount=("discount_pct", "mean"),
        avg_demand=("demand_qty", "mean")
    )
    .round(2)
)

print("\n===== Promotion 여부 비교 =====")
display(promotion_summary)


# ------------------------------------------------------------
# 4. Discount 구간별 평균 Demand
# ------------------------------------------------------------

discount_summary = (
    eda_df
    .groupby("discount_pct")
    .agg(
        count=("demand_qty", "size"),
        avg_price=("price", "mean"),
        avg_demand=("demand_qty", "mean"),
        promotion_rate=("promotion_flag", "mean")
    )
    .round(2)
)

print("\n===== 할인율별 평균 Demand =====")
display(discount_summary)


# ------------------------------------------------------------
# 5. 산점도
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.scatter(
    eda_df["price"],
    eda_df["demand_qty"],
    alpha=0.15
)

plt.xlabel("Price")
plt.ylabel("Demand")
plt.title("Price vs Demand")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# [Rolling Validation] 최종 Point 모델과 동일한 2000회 설정
# ============================================================

rolling_splits = [
    {"name": "Fold 1", "train_end": "2023-04-30", "valid_start": "2023-05-01", "valid_end": "2023-06-30"},
    {"name": "Fold 2", "train_end": "2023-06-30", "valid_start": "2023-07-01", "valid_end": "2023-08-31"},
    {"name": "Fold 3", "train_end": "2023-08-31", "valid_start": "2023-09-01", "valid_end": "2023-10-31"},
]

rolling_results = []

for split in rolling_splits:
    fold_train = model_df[
        model_df["date"] <= pd.Timestamp(split["train_end"])
    ].copy()
    fold_valid = model_df[
        model_df["date"].between(
            pd.Timestamp(split["valid_start"]),
            pd.Timestamp(split["valid_end"]),
        )
    ].copy()

    X_fold_train = fold_train[feature_columns].copy()
    y_fold_train = fold_train[target].copy()
    X_fold_valid = fold_valid[feature_columns].copy()
    y_fold_valid = fold_valid[target].copy()
    fold_cat_indices = [
        X_fold_train.columns.get_loc(col) for col in categorical_features
    ]

    fold_model = CatBoostRegressor(**FINAL_POINT_PARAMS)
    fold_model.fit(
        X_fold_train,
        y_fold_train,
        cat_features=fold_cat_indices,
        eval_set=(X_fold_valid, y_fold_valid),
        early_stopping_rounds=150,
        verbose=False,
    )

    fold_pred = np.maximum(fold_model.predict(X_fold_valid), 0)
    actual = y_fold_valid.to_numpy()
    rolling_results.append({
        "Fold": split["name"],
        "Train_End": split["train_end"],
        "Valid_Start": split["valid_start"],
        "Valid_End": split["valid_end"],
        "MAE": mean_absolute_error(actual, fold_pred),
        "RMSE": np.sqrt(mean_squared_error(actual, fold_pred)),
        "R2": r2_score(actual, fold_pred),
        "WAPE": np.abs(actual - fold_pred).sum() / np.abs(actual).sum() * 100,
        "Bias": np.mean(fold_pred - actual),
        "Best_Iteration": fold_model.get_best_iteration(),
        "Configured_Iterations": FINAL_POINT_PARAMS["iterations"],
    })

rolling_result_df = pd.DataFrame(rolling_results)
display(rolling_result_df.round(4))
print("평균 R²:", round(rolling_result_df["R2"].mean(), 4))
print("평균 WAPE:", round(rolling_result_df["WAPE"].mean(), 4), "%")

## 4. Quantile Forecast·Calibration

In [ ]:
# ============================================================
# [V2 셀 18] P90 / P95 Quantile CatBoost
# ============================================================

# ------------------------------------------------------------
# P90 모델
# ------------------------------------------------------------

p90_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=7,

    loss_function="Quantile:alpha=0.90",

    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=False
)


p90_model.fit(
    X_train,
    y_train,

    cat_features=cat_indices,

    eval_set=(
        X_valid,
        y_valid
    ),

    verbose=False
)


# ------------------------------------------------------------
# P95 모델
# ------------------------------------------------------------

p95_model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.05,
    depth=7,

    loss_function="Quantile:alpha=0.95",

    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=False
)


p95_model.fit(
    X_train,
    y_train,

    cat_features=cat_indices,

    eval_set=(
        X_valid,
        y_valid
    ),

    verbose=False
)


# ------------------------------------------------------------
# Validation 예측
# ------------------------------------------------------------

p90_pred = np.maximum(
    p90_model.predict(X_valid),
    0
)

p95_pred = np.maximum(
    p95_model.predict(X_valid),
    0
)


# ------------------------------------------------------------
# Quantile 평가 함수
# ------------------------------------------------------------

def evaluate_inventory_quantile(
    y_true,
    target_stock,
    name
):

    y_true = np.asarray(y_true)
    target_stock = np.asarray(target_stock)


    service_level = np.mean(
        y_true <= target_stock
    ) * 100


    stockout_rate = np.mean(
        y_true > target_stock
    ) * 100


    shortage = np.maximum(
        y_true - target_stock,
        0
    )


    remaining = np.maximum(
        target_stock - y_true,
        0
    )


    fill_rate = (
        1
        -
        shortage.sum()
        /
        y_true.sum()
    ) * 100


    print(
        f"===== {name} ====="
    )

    print(
        f"Service Level : {service_level:.2f}%"
    )

    print(
        f"Stockout Rate : {stockout_rate:.2f}%"
    )

    print(
        f"Fill Rate     : {fill_rate:.2f}%"
    )

    print(
        f"Avg Target    : {target_stock.mean():.2f}"
    )

    print(
        f"Avg Remaining : {remaining.mean():.2f}"
    )

    print(
        f"Total Shortage: {shortage.sum():.2f}"
    )

    print()


# ------------------------------------------------------------
# 평가
# ------------------------------------------------------------

evaluate_inventory_quantile(
    y_valid.values,
    p90_pred,
    "V2 P90"
)


evaluate_inventory_quantile(
    y_valid.values,
    p95_pred,
    "V2 P95"
)

In [ ]:
# ============================================================
# [Quantile Calibration] Validation에서 계산하고 변수로 유지
# ============================================================

y_valid_array = y_valid.to_numpy()
p90_error = y_valid_array - p90_pred
p95_error = y_valid_array - p95_pred

p90_correction = float(np.quantile(p90_error, 0.90))
p95_correction = float(np.quantile(p95_error, 0.95))

p90_calibrated = np.maximum(p90_pred + p90_correction, 0)
p95_calibrated_raw = np.maximum(p95_pred + p95_correction, 0)

# 단조성 규칙은 Test를 보기 전에 Validation 단계에서 고정한다.
p95_calibrated = np.maximum(p95_calibrated_raw, p90_calibrated)

print("P90 correction:", repr(p90_correction))
print("P95 correction:", repr(p95_correction))
print("Validation crossing 수정 건수:", (p95_calibrated_raw < p90_calibrated).sum())

evaluate_inventory_quantile(y_valid_array, p90_calibrated, "V2 P90 Calibrated")
evaluate_inventory_quantile(y_valid_array, p95_calibrated, "V2 P95 Calibrated + Monotonic")

print("P90 pinball loss:", mean_pinball_loss(y_valid_array, p90_calibrated, alpha=0.90))
print("P95 pinball loss:", mean_pinball_loss(y_valid_array, p95_calibrated, alpha=0.95))

In [ ]:
# ============================================================
# [Quantile Rolling Validation]
# 각 Fold의 과거 마지막 30일로 보정한 뒤 미래 구간을 평가
# ============================================================

QUANTILE_CALIBRATION_DAYS = 30
quantile_rolling_results = []

for split in rolling_splits:
    train_end_ts = pd.Timestamp(split["train_end"])
    calibration_start = train_end_ts - pd.Timedelta(
        days=QUANTILE_CALIBRATION_DAYS - 1
    )

    fold_fit = model_df[model_df["date"] < calibration_start].copy()
    fold_calibration = model_df[
        model_df["date"].between(calibration_start, train_end_ts)
    ].copy()
    fold_evaluation = model_df[
        model_df["date"].between(
            pd.Timestamp(split["valid_start"]),
            pd.Timestamp(split["valid_end"]),
        )
    ].copy()

    X_fit = fold_fit[feature_columns]
    y_fit = fold_fit[target]
    X_calibration = fold_calibration[feature_columns]
    y_calibration = fold_calibration[target].to_numpy()
    X_evaluation = fold_evaluation[feature_columns]
    y_evaluation = fold_evaluation[target].to_numpy()

    fold_cat_indices = [
        X_fit.columns.get_loc(col) for col in categorical_features
    ]

    fold_p90_model = CatBoostRegressor(
        **FINAL_QUANTILE_PARAMS,
        loss_function="Quantile:alpha=0.90",
    )
    fold_p95_model = CatBoostRegressor(
        **FINAL_QUANTILE_PARAMS,
        loss_function="Quantile:alpha=0.95",
    )

    fold_p90_model.fit(
        X_fit, y_fit, cat_features=fold_cat_indices, verbose=False
    )
    fold_p95_model.fit(
        X_fit, y_fit, cat_features=fold_cat_indices, verbose=False
    )

    cal_p90_raw = np.maximum(fold_p90_model.predict(X_calibration), 0)
    cal_p95_raw = np.maximum(fold_p95_model.predict(X_calibration), 0)
    fold_p90_correction = float(np.quantile(y_calibration - cal_p90_raw, 0.90))
    fold_p95_correction = float(np.quantile(y_calibration - cal_p95_raw, 0.95))

    eval_p90 = np.maximum(
        fold_p90_model.predict(X_evaluation) + fold_p90_correction, 0
    )
    eval_p95_raw = np.maximum(
        fold_p95_model.predict(X_evaluation) + fold_p95_correction, 0
    )
    crossing_before_fix = eval_p95_raw < eval_p90
    eval_p95 = np.maximum(eval_p95_raw, eval_p90)

    quantile_rolling_results.append({
        "Fold": split["name"],
        "Fit_End": str((calibration_start - pd.Timedelta(days=1)).date()),
        "Calibration_Start": str(calibration_start.date()),
        "Calibration_End": split["train_end"],
        "Evaluation_Start": split["valid_start"],
        "Evaluation_End": split["valid_end"],
        "P90_Coverage": np.mean(y_evaluation <= eval_p90) * 100,
        "P95_Coverage": np.mean(y_evaluation <= eval_p95) * 100,
        "P90_Pinball": mean_pinball_loss(y_evaluation, eval_p90, alpha=0.90),
        "P95_Pinball": mean_pinball_loss(y_evaluation, eval_p95, alpha=0.95),
        "P90_Correction": fold_p90_correction,
        "P95_Correction": fold_p95_correction,
        "Crossing_Rate_Before_Fix": crossing_before_fix.mean() * 100,
    })

quantile_rolling_result_df = pd.DataFrame(quantile_rolling_results)
display(quantile_rolling_result_df.round(4))
print(
    "평균 P90 coverage:",
    round(quantile_rolling_result_df["P90_Coverage"].mean(), 2),
    "%",
)
print(
    "평균 P95 coverage:",
    round(quantile_rolling_result_df["P95_Coverage"].mean(), 2),
    "%",
)

In [ ]:
# ============================================================
# [최종 Point 모델] Train + Validation -> Test
# ============================================================

final_train_df = pd.concat([train_df, valid_df], axis=0).sort_values(
    "date"
).reset_index(drop=True)

X_final_train = final_train_df[feature_columns].copy()
y_final_train = final_train_df[target].copy()
X_test = test_df[feature_columns].copy()
y_test = test_df[target].copy()

final_cat_indices = [
    X_final_train.columns.get_loc(col) for col in categorical_features
]

final_point_model = CatBoostRegressor(**FINAL_POINT_PARAMS)
final_point_model.fit(
    X_final_train,
    y_final_train,
    cat_features=final_cat_indices,
    verbose=200,
)

test_point_pred = np.maximum(final_point_model.predict(X_test), 0)
final_point_result = evaluate_regression(
    y_test.to_numpy(), test_point_pred, "V2 Final Point Model - TEST"
)

In [ ]:
# ============================================================
# [V2 셀 ADD 20-1] Test 기간 실제 수요 vs 예측 수요 시각화
# ============================================================

viz_df = pd.DataFrame({
    "date": test_df["date"].values,
    "actual": y_test.values,
    "predicted": test_point_pred
})

# 날짜별 평균으로 집계 (매장 x 상품이 여러 개라 하루에 여러 행 존재)
daily_viz = (
    viz_df
    .groupby("date")[["actual", "predicted"]]
    .mean()
    .reset_index()
)

# ------------------------------------------------------------
# 1. 꺾은선 그래프 (Test 기간 실제 vs 예측 추이)
# ------------------------------------------------------------

plt.figure(figsize=(15, 5))

plt.plot(daily_viz["date"], daily_viz["actual"], label="Actual")
plt.plot(daily_viz["date"], daily_viz["predicted"], label="Predicted")

plt.axvline(
    test_df["date"].min(),
    color="tab:blue",
    linestyle="--",
    label="Test Start"
)

plt.title("Test 기간 실제 수요 vs 예측 수요")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. 산점도 (Actual vs Predicted, 45도 기준선)
# ------------------------------------------------------------

plt.figure(figsize=(6, 6))

plt.scatter(
    viz_df["actual"],
    viz_df["predicted"],
    alpha=0.3,
    s=15
)

lims = [
    min(viz_df["actual"].min(), viz_df["predicted"].min()),
    max(viz_df["actual"].max(), viz_df["predicted"].max())
]
plt.plot(lims, lims, color="red", linestyle="--", label="y = x")

plt.title("Actual vs Predicted (Test)")
plt.xlabel("Actual Demand")
plt.ylabel("Predicted Demand")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# [최종 Quantile 모델] 보정 + 단조성까지 하나의 고정 파이프라인
# ============================================================

P90_CORRECTION = float(p90_correction)
P95_CORRECTION = float(p95_correction)

final_p90_model = CatBoostRegressor(
    **FINAL_QUANTILE_PARAMS,
    loss_function="Quantile:alpha=0.90",
)
final_p95_model = CatBoostRegressor(
    **FINAL_QUANTILE_PARAMS,
    loss_function="Quantile:alpha=0.95",
)

final_p90_model.fit(
    X_final_train, y_final_train,
    cat_features=final_cat_indices, verbose=200,
)
final_p95_model.fit(
    X_final_train, y_final_train,
    cat_features=final_cat_indices, verbose=200,
)

test_p90_raw = np.maximum(final_p90_model.predict(X_test), 0)
test_p95_raw = np.maximum(final_p95_model.predict(X_test), 0)

test_p90_calibrated = np.maximum(test_p90_raw + P90_CORRECTION, 0)
test_p95_calibrated_raw = np.maximum(test_p95_raw + P95_CORRECTION, 0)
test_crossing_mask = test_p95_calibrated_raw < test_p90_calibrated
test_p95_calibrated = np.maximum(
    test_p95_calibrated_raw,
    test_p90_calibrated,
)

print("Test crossing 수정 건수:", test_crossing_mask.sum())
print("Test crossing 비율:", round(test_crossing_mask.mean() * 100, 2), "%")

evaluate_inventory_quantile(
    y_test.to_numpy(), test_p90_calibrated,
    "V2 Final P90 Calibrated - TEST",
)
evaluate_inventory_quantile(
    y_test.to_numpy(), test_p95_calibrated,
    "V2 Final P95 Calibrated + Monotonic - TEST",
)

print("P90 Test pinball loss:", mean_pinball_loss(y_test, test_p90_calibrated, alpha=0.90))
print("P95 Test pinball loss:", mean_pinball_loss(y_test, test_p95_calibrated, alpha=0.95))

In [ ]:
# ============================================================
# [Test 세부 진단] SKU 성능과 Quantile 구간별 Coverage
# ============================================================

test_diagnostics_df = test_df[
    ["date", "store_id", "product_id", "category", "promotion_flag", "demand_qty"]
].copy()
test_diagnostics_df["point_forecast"] = test_point_pred
test_diagnostics_df["p90"] = test_p90_calibrated
test_diagnostics_df["p95"] = test_p95_calibrated


def summarize_point_group(group):
    actual = group["demand_qty"].to_numpy()
    predicted = group["point_forecast"].to_numpy()
    denominator = np.abs(actual).sum()
    return pd.Series({
        "Rows": len(group),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted) if np.unique(actual).size > 1 else np.nan,
        "WAPE": np.abs(actual - predicted).sum() / denominator * 100 if denominator else np.nan,
        "Bias": np.mean(predicted - actual),
    })


point_series_rows = []
for (store_id, product_id), group in test_diagnostics_df.groupby(
    ["store_id", "product_id"], observed=True
):
    summary = summarize_point_group(group).to_dict()
    summary["store_id"] = store_id
    summary["product_id"] = product_id
    point_series_rows.append(summary)

point_series_performance_df = pd.DataFrame(point_series_rows)[
    ["store_id", "product_id", "Rows", "MAE", "RMSE", "R2", "WAPE", "Bias"]
]

coverage_rows = []
for dimension in ["store_id", "category", "promotion_flag"]:
    for value, group in test_diagnostics_df.groupby(dimension, observed=True):
        actual = group["demand_qty"].to_numpy()
        coverage_rows.append({
            "Dimension": dimension,
            "Value": str(value),
            "Rows": len(group),
            "P90_Coverage": np.mean(actual <= group["p90"].to_numpy()) * 100,
            "P95_Coverage": np.mean(actual <= group["p95"].to_numpy()) * 100,
            "P90_Pinball": mean_pinball_loss(actual, group["p90"], alpha=0.90),
            "P95_Pinball": mean_pinball_loss(actual, group["p95"], alpha=0.95),
        })

quantile_slice_coverage_df = pd.DataFrame(coverage_rows)

print("===== 매장×상품별 Point 성능 =====")
display(point_series_performance_df.sort_values("WAPE", ascending=False).head(20).round(4))
print("Macro WAPE:", round(point_series_performance_df["WAPE"].mean(), 4), "%")

print("\n===== Quantile 구간별 Coverage =====")
display(quantile_slice_coverage_df.round(4))

In [ ]:
# ============================================================
# [발주 추천] 재고 포지션·입고예정·MOQ·박스 단위 반영
# ============================================================

# 실제 운영 마스터가 없으므로 기본값은 제약 없음이다.
# Units Ordered는 입고예정일이 없는 과거 컬럼이므로 open order로 재사용하지 않는다.
ORDER_INPUT_DEFAULTS = {
    "incoming_order_qty": 0.0,
    "backorder_qty": 0.0,
    "pack_size": 1,
    "minimum_order_qty": 0,
    "initial_incoming_day_1": 0.0,
    "initial_incoming_day_2": 0.0,
    "initial_incoming_day_3": 0.0,
    "initial_incoming_day_4": 0.0,
}

# 실제 원가가 입력되기 전에는 비용 순위를 계산하지 않는다.
COST_SCENARIO = {
    "currency": None,
    "fixed_order_cost": None,
    "unit_purchase_cost": None,
    "unit_holding_cost_per_day": None,
    "unit_shortage_cost": None,
}


def ensure_order_inputs(frame):
    result = frame.copy()
    for column, default in ORDER_INPUT_DEFAULTS.items():
        if column not in result.columns:
            result[column] = default
        else:
            result[column] = result[column].fillna(default)

    numeric_columns = list(ORDER_INPUT_DEFAULTS)
    for column in numeric_columns:
        result[column] = pd.to_numeric(result[column], errors="raise")

    nonnegative_columns = [
        "incoming_order_qty",
        "backorder_qty",
        "minimum_order_qty",
        "initial_incoming_day_1",
        "initial_incoming_day_2",
        "initial_incoming_day_3",
        "initial_incoming_day_4",
    ]
    if result[nonnegative_columns].lt(0).any().any():
        raise ValueError("입고예정·미납·MOQ 수량은 0 이상이어야 합니다.")
    if result["pack_size"].le(0).any():
        raise ValueError("pack_size는 1 이상의 양수여야 합니다.")
    return result


def calculate_constrained_order_qty(
    target_stock,
    on_hand,
    incoming_order_qty=0,
    backorder_qty=0,
    pack_size=1,
    minimum_order_qty=0,
):
    target = np.asarray(target_stock, dtype=float)
    hand = np.asarray(on_hand, dtype=float)
    incoming = np.asarray(incoming_order_qty, dtype=float)
    backorder = np.asarray(backorder_qty, dtype=float)
    pack = np.asarray(pack_size, dtype=float)
    moq = np.asarray(minimum_order_qty, dtype=float)

    if np.any(pack <= 0):
        raise ValueError("pack_size는 1 이상의 양수여야 합니다.")
    if np.any(incoming < 0) or np.any(backorder < 0) or np.any(moq < 0):
        raise ValueError("입고예정·미납·MOQ 수량은 0 이상이어야 합니다.")

    inventory_position = hand + incoming - backorder
    raw_order = np.maximum(target - inventory_position, 0)
    pack_rounded = np.where(
        raw_order > 0,
        np.ceil(raw_order / pack) * pack,
        0,
    )
    moq_rounded = np.ceil(moq / pack) * pack
    constrained = np.where(
        pack_rounded > 0,
        np.maximum(pack_rounded, moq_rounded),
        0,
    )
    return constrained.astype(int)


def evaluate_cost_scenario(simulation_detail, assumptions):
    required = [
        "fixed_order_cost",
        "unit_purchase_cost",
        "unit_holding_cost_per_day",
        "unit_shortage_cost",
    ]
    missing = [key for key in required if assumptions.get(key) is None]
    if missing:
        raise ValueError(f"비용 가정 누락: {missing}")
    if any(float(assumptions[key]) < 0 for key in required):
        raise ValueError("비용 가정은 0 이상이어야 합니다.")

    detail = simulation_detail.copy()
    detail["fixed_order_cost"] = (
        detail["order_qty"].gt(0).astype(float)
        * float(assumptions["fixed_order_cost"])
    )
    detail["purchase_cost"] = (
        detail["order_qty"] * float(assumptions["unit_purchase_cost"])
    )
    detail["holding_cost"] = (
        detail["ending_inventory"]
        * float(assumptions["unit_holding_cost_per_day"])
    )
    detail["shortage_cost"] = (
        detail["shortage_qty"] * float(assumptions["unit_shortage_cost"])
    )
    cost_columns = [
        "fixed_order_cost",
        "purchase_cost",
        "holding_cost",
        "shortage_cost",
    ]
    detail["total_cost"] = detail[cost_columns].sum(axis=1)
    summary = (
        detail.groupby("policy")[cost_columns + ["total_cost"]]
        .sum()
        .reset_index()
    )
    summary["currency"] = assumptions.get("currency")
    return detail, summary


# 단위 검증: 입고예정 차감, MOQ 상향, 박스 배수 반올림.
assert int(calculate_constrained_order_qty(100, 60, 30, 0, 12, 24)) == 24
assert int(calculate_constrained_order_qty(100, 110, 0, 0, 12, 24)) == 0

order_df = ensure_order_inputs(test_df)
order_df["point_forecast"] = test_point_pred
order_df["p90_target_stock"] = test_p90_calibrated
order_df["p95_target_stock_raw"] = test_p95_calibrated_raw
order_df["p95_target_stock_fixed"] = test_p95_calibrated
order_df["inventory_position"] = (
    order_df["inventory_level"]
    + order_df["incoming_order_qty"]
    - order_df["backorder_qty"]
)

order_df["p90_order_qty_unconstrained"] = np.ceil(np.maximum(
    order_df["p90_target_stock"] - order_df["inventory_position"], 0
)).astype(int)
order_df["p95_order_qty_unconstrained"] = np.ceil(np.maximum(
    order_df["p95_target_stock_fixed"] - order_df["inventory_position"], 0
)).astype(int)
order_df["p90_order_qty"] = calculate_constrained_order_qty(
    order_df["p90_target_stock"],
    order_df["inventory_level"],
    order_df["incoming_order_qty"],
    order_df["backorder_qty"],
    order_df["pack_size"],
    order_df["minimum_order_qty"],
)
order_df["p95_order_qty_fixed"] = calculate_constrained_order_qty(
    order_df["p95_target_stock_fixed"],
    order_df["inventory_level"],
    order_df["incoming_order_qty"],
    order_df["backorder_qty"],
    order_df["pack_size"],
    order_df["minimum_order_qty"],
)
order_df["order_constraint_applied"] = (
    order_df["p95_order_qty_fixed"]
    != order_df["p95_order_qty_unconstrained"]
)
order_df["p90_order_needed"] = order_df["p90_order_qty"] > 0
order_df["p95_order_needed"] = order_df["p95_order_qty_fixed"] > 0

print("P90 총 추천 발주량:", order_df["p90_order_qty"].sum())
print("P95 총 추천 발주량:", order_df["p95_order_qty_fixed"].sum())
print("P95 발주 필요 비율:", round(order_df["p95_order_needed"].mean() * 100, 2), "%")
print("제약 적용 행 수:", int(order_df["order_constraint_applied"].sum()))

In [ ]:
# ============================================================
# [Protection-period demand model + leakage-safe rolling conformal calibration]
# The 5-day target and planned features are purged at split boundaries.
# Calibration-window selection uses validation only; test is evaluated once afterward.
# ============================================================

LEAD_TIME_DAYS = 4
REVIEW_PERIOD_DAYS = 1
PROTECTION_PERIOD_DAYS = LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS
LEADTIME_TARGET = "protection_period_demand"

LEADTIME_REQUIRED_PLAN_COLUMNS = [
    "price", "discount_pct", "promotion_flag", "is_holiday",
]
LEADTIME_AGGREGATION_RULES = {
    "protection_price_mean": {"source": "price", "aggregation": "mean"},
    "protection_discount_mean": {"source": "discount_pct", "aggregation": "mean"},
    "protection_promotion_days": {"source": "promotion_flag", "aggregation": "sum"},
    "protection_holiday_days": {"source": "is_holiday", "aggregation": "sum"},
}
LEADTIME_FUTURE_FEATURES = list(LEADTIME_AGGREGATION_RULES)
leadtime_feature_columns = feature_columns + LEADTIME_FUTURE_FEATURES


def forward_aggregate(series, window, aggregation):
    reversed_series = series.iloc[::-1]
    rolling = reversed_series.rolling(window=window, min_periods=window)
    if aggregation == "sum":
        values = rolling.sum()
    elif aggregation == "mean":
        values = rolling.mean()
    else:
        raise ValueError(f"Unsupported aggregation: {aggregation}")
    return values.iloc[::-1]


leadtime_model_df = model_df.copy()
leadtime_model_df[LEADTIME_TARGET] = (
    leadtime_model_df.groupby(["store_id", "product_id"])["demand_qty"]
    .transform(lambda x: forward_aggregate(x, PROTECTION_PERIOD_DAYS, "sum"))
)
for feature_name, rule in LEADTIME_AGGREGATION_RULES.items():
    leadtime_model_df[feature_name] = (
        leadtime_model_df.groupby(["store_id", "product_id"])[rule["source"]]
        .transform(
            lambda x, aggregation=rule["aggregation"]: forward_aggregate(
                x, PROTECTION_PERIOD_DAYS, aggregation
            )
        )
    )
leadtime_model_df["protection_end_date"] = (
    leadtime_model_df["date"]
    + pd.to_timedelta(PROTECTION_PERIOD_DAYS - 1, unit="D")
)

# Purge rows whose cumulative target or plan crosses a split boundary.
leadtime_train_df = leadtime_model_df[
    leadtime_model_df["protection_end_date"] <= train_end
].dropna(subset=[LEADTIME_TARGET] + LEADTIME_FUTURE_FEATURES).copy()
leadtime_calibration_df = leadtime_model_df[
    (leadtime_model_df["date"] > train_end)
    & (leadtime_model_df["protection_end_date"] <= valid_end)
].dropna(subset=[LEADTIME_TARGET] + LEADTIME_FUTURE_FEATURES).copy()
leadtime_test_eval_df = leadtime_model_df[
    leadtime_model_df["date"] > valid_end
].dropna(subset=[LEADTIME_TARGET] + LEADTIME_FUTURE_FEATURES).copy()

X_leadtime_train = leadtime_train_df[leadtime_feature_columns]
y_leadtime_train = leadtime_train_df[LEADTIME_TARGET]
X_leadtime_calibration = leadtime_calibration_df[leadtime_feature_columns]
X_leadtime_test_eval = leadtime_test_eval_df[leadtime_feature_columns]
y_leadtime_test_eval = leadtime_test_eval_df[LEADTIME_TARGET].to_numpy()
if X_leadtime_train.isna().any().any():
    raise ValueError("Protection-period training features contain missing values.")

leadtime_cat_indices = [
    X_leadtime_train.columns.get_loc(col) for col in categorical_features
]
leadtime_p90_model = CatBoostRegressor(
    **FINAL_QUANTILE_PARAMS, loss_function="Quantile:alpha=0.90"
)
leadtime_p95_model = CatBoostRegressor(
    **FINAL_QUANTILE_PARAMS, loss_function="Quantile:alpha=0.95"
)
leadtime_p90_model.fit(
    X_leadtime_train, y_leadtime_train,
    cat_features=leadtime_cat_indices, verbose=200,
)
leadtime_p95_model.fit(
    X_leadtime_train, y_leadtime_train,
    cat_features=leadtime_cat_indices, verbose=200,
)


def score_leadtime_rows(frame, feature_frame):
    scored = frame.copy()
    scored["raw_p90"] = np.maximum(
        leadtime_p90_model.predict(feature_frame), 0
    )
    scored["raw_p95"] = np.maximum(
        leadtime_p95_model.predict(feature_frame), 0
    )
    scored["residual_p90"] = scored[LEADTIME_TARGET] - scored["raw_p90"]
    scored["residual_p95"] = scored[LEADTIME_TARGET] - scored["raw_p95"]
    return scored


def online_leadtime_calibrate(seed, evaluation, window_days):
    """Prequential calibration: use only residuals matured before each forecast date."""
    history = seed[
        ["date", "protection_end_date", "residual_p90", "residual_p95"]
    ].copy()
    pieces = []
    for forecast_date, current in evaluation.groupby("date", sort=True):
        eligible = history[history["protection_end_date"] < forecast_date]
        if window_days is not None:
            eligible = eligible[
                eligible["protection_end_date"]
                >= forecast_date - pd.Timedelta(days=window_days)
            ]
        if eligible.empty:
            raise ValueError(f"No matured calibration residual at {forecast_date}")
        correction_p90 = float(np.quantile(eligible["residual_p90"], 0.90))
        correction_p95 = float(np.quantile(eligible["residual_p95"], 0.95))
        output = current.copy()
        output["correction_p90"] = correction_p90
        output["correction_p95"] = correction_p95
        output["calibration_rows"] = len(eligible)
        output["pred_p90"] = np.maximum(output["raw_p90"] + correction_p90, 0)
        p95_before_crossing = np.maximum(
            output["raw_p95"] + correction_p95, 0
        )
        output["pred_p95"] = np.maximum(
            p95_before_crossing, output["pred_p90"]
        )
        pieces.append(output)
        history = pd.concat([
            history,
            current[[
                "date", "protection_end_date", "residual_p90", "residual_p95"
            ]],
        ], ignore_index=True)
    return pd.concat(pieces).sort_index()


def leadtime_metrics(frame):
    actual = frame[LEADTIME_TARGET].to_numpy()
    p90 = frame["pred_p90"].to_numpy()
    p95 = frame["pred_p95"].to_numpy()
    return {
        "Rows": len(frame),
        "P90_Coverage": float(np.mean(actual <= p90) * 100),
        "P95_Coverage": float(np.mean(actual <= p95) * 100),
        "P90_Pinball": float(mean_pinball_loss(actual, p90, alpha=0.90)),
        "P95_Pinball": float(mean_pinball_loss(actual, p95, alpha=0.95)),
        "Crossing_Count": int(np.sum(p95 < p90)),
        "Avg_P90_Correction": float(frame["correction_p90"].mean()),
        "Avg_P95_Correction": float(frame["correction_p95"].mean()),
    }


# Fixed correction is retained as a documented fallback and comparison baseline.
leadtime_calibration_scored = score_leadtime_rows(
    leadtime_calibration_df, X_leadtime_calibration
)
leadtime_test_scored = score_leadtime_rows(
    leadtime_test_eval_df, X_leadtime_test_eval
)
LEADTIME_P90_CORRECTION = float(np.quantile(
    leadtime_calibration_scored["residual_p90"], 0.90
))
LEADTIME_P95_CORRECTION = float(np.quantile(
    leadtime_calibration_scored["residual_p95"], 0.95
))

# August seeds the online calibration. September-October select the window.
# The test target is not inspected during this choice.
LEADTIME_VALIDATION_SEED_END = pd.Timestamp("2023-08-31")
LEADTIME_WINDOW_CANDIDATES = [14, 21, 30, 45, 60, None]
leadtime_validation_seed = leadtime_calibration_scored[
    leadtime_calibration_scored["protection_end_date"]
    <= LEADTIME_VALIDATION_SEED_END
].copy()
leadtime_validation_eval = leadtime_calibration_scored[
    leadtime_calibration_scored["date"] > LEADTIME_VALIDATION_SEED_END
].copy()

selection_rows = []
for candidate_window in LEADTIME_WINDOW_CANDIDATES:
    candidate_output = online_leadtime_calibrate(
        leadtime_validation_seed, leadtime_validation_eval, candidate_window
    )
    candidate_metrics = leadtime_metrics(candidate_output)
    candidate_metrics["Window"] = (
        "expanding" if candidate_window is None else str(candidate_window)
    )
    candidate_metrics["Coverage_Error"] = (
        abs(candidate_metrics["P90_Coverage"] - 90)
        + abs(candidate_metrics["P95_Coverage"] - 95)
    )
    candidate_metrics["Selection_Score"] = (
        candidate_metrics["Coverage_Error"]
        + 0.01
        * (candidate_metrics["P90_Pinball"] + candidate_metrics["P95_Pinball"])
    )
    selection_rows.append(candidate_metrics)

leadtime_calibration_selection = pd.DataFrame(selection_rows).sort_values(
    ["Selection_Score", "P95_Pinball", "P90_Pinball"]
).reset_index(drop=True)
selected_window_label = str(leadtime_calibration_selection.loc[0, "Window"])
LEADTIME_CALIBRATION_WINDOW_DAYS = (
    None if selected_window_label == "expanding" else int(selected_window_label)
)

# Test is opened once, after selection. Residuals join history only after the
# full 5-day target has matured, preventing look-ahead leakage.
leadtime_test_rolling = online_leadtime_calibrate(
    leadtime_calibration_scored,
    leadtime_test_scored,
    LEADTIME_CALIBRATION_WINDOW_DAYS,
)
leadtime_test_fixed = leadtime_test_scored.copy()
leadtime_test_fixed["correction_p90"] = LEADTIME_P90_CORRECTION
leadtime_test_fixed["correction_p95"] = LEADTIME_P95_CORRECTION
leadtime_test_fixed["pred_p90"] = np.maximum(
    leadtime_test_fixed["raw_p90"] + LEADTIME_P90_CORRECTION, 0
)
leadtime_test_fixed["pred_p95"] = np.maximum(
    np.maximum(
        leadtime_test_fixed["raw_p95"] + LEADTIME_P95_CORRECTION, 0
    ),
    leadtime_test_fixed["pred_p90"],
)

leadtime_model_evaluation = pd.DataFrame([
    {
        "Method": f"rolling_{selected_window_label}_days_validation_selected",
        "Protection_Period_Days": PROTECTION_PERIOD_DAYS,
        "Feature_Count": len(leadtime_feature_columns),
        "Planned_Feature_Count": len(LEADTIME_FUTURE_FEATURES),
        **leadtime_metrics(leadtime_test_rolling),
    },
    {
        "Method": "fixed_full_validation_fallback",
        "Protection_Period_Days": PROTECTION_PERIOD_DAYS,
        "Feature_Count": len(leadtime_feature_columns),
        "Planned_Feature_Count": len(LEADTIME_FUTURE_FEATURES),
        **leadtime_metrics(leadtime_test_fixed),
    },
])
leadtime_test_p90 = leadtime_test_rolling["pred_p90"].to_numpy()
leadtime_test_p95 = leadtime_test_rolling["pred_p95"].to_numpy()

# Persist all realized residuals. In live operation this file must be refreshed
# after the 5-day target matures; inference reports stale calibration explicitly.
leadtime_residual_history = pd.concat([
    leadtime_calibration_scored,
    leadtime_test_scored,
], ignore_index=True)[[
    "date", "protection_end_date", "store_id", "product_id",
    LEADTIME_TARGET, "raw_p90", "raw_p95", "residual_p90", "residual_p95",
]].sort_values(["protection_end_date", "store_id", "product_id"])


def get_rolling_leadtime_corrections(
    forecast_dates,
    residual_history=leadtime_residual_history,
    window_days=LEADTIME_CALIBRATION_WINDOW_DAYS,
):
    corrections = []
    for forecast_date in pd.to_datetime(pd.Series(forecast_dates)):
        eligible = residual_history[
            residual_history["protection_end_date"] < forecast_date
        ]
        if eligible.empty:
            corrections.append((
                LEADTIME_P90_CORRECTION, LEADTIME_P95_CORRECTION,
                pd.NaT, np.nan, "fallback_no_matured_residual",
            ))
            continue
        recent = eligible
        if window_days is not None:
            recent = eligible[
                eligible["protection_end_date"]
                >= forecast_date - pd.Timedelta(days=window_days)
            ]
        status = "fresh"
        if recent.empty:
            latest_date = eligible["protection_end_date"].max()
            recent = eligible[
                eligible["protection_end_date"]
                >= latest_date - pd.Timedelta(days=window_days - 1)
            ]
            status = "stale_fallback_latest_window"
        as_of = recent["protection_end_date"].max()
        age_days = int((forecast_date - as_of).days)
        if age_days > 14:
            status = "stale_fallback_latest_window"
        corrections.append((
            float(np.quantile(recent["residual_p90"], 0.90)),
            float(np.quantile(recent["residual_p95"], 0.95)),
            as_of,
            age_days,
            status,
        ))
    return pd.DataFrame(corrections, columns=[
        "p90_correction", "p95_correction", "calibration_as_of",
        "calibration_age_days", "calibration_status",
    ])


display(leadtime_calibration_selection.round(4))
display(leadtime_model_evaluation.round(4))

# Map prequential test targets to the operational order frame.
X_leadtime_test_all = leadtime_model_df.loc[X_test.index, leadtime_feature_columns]
leadtime_plan_ready = X_leadtime_test_all.notna().all(axis=1)
order_df["leadtime_plan_ready"] = leadtime_plan_ready.to_numpy()
order_df["leadtime_p90_target_stock"] = np.nan
order_df["leadtime_p95_target_stock"] = np.nan
ready_index = X_leadtime_test_all.index[leadtime_plan_ready]
if not ready_index.equals(leadtime_test_rolling.index):
    raise ValueError("Lead-time evaluation index does not match plan-ready rows.")
order_df.loc[
    order_df["leadtime_plan_ready"], "leadtime_p90_target_stock"
] = leadtime_test_rolling.loc[ready_index, "pred_p90"].to_numpy()
order_df.loc[
    order_df["leadtime_plan_ready"], "leadtime_p95_target_stock"
] = leadtime_test_rolling.loc[ready_index, "pred_p95"].to_numpy()

assert (
    order_df.loc[order_df["leadtime_plan_ready"], "leadtime_p95_target_stock"]
    >= order_df.loc[order_df["leadtime_plan_ready"], "leadtime_p90_target_stock"]
).all()
print("Selected rolling calibration window:", selected_window_label)
print("Protection-period planned features:", LEADTIME_FUTURE_FEATURES)
print(
    "Simulation-ready period:",
    order_df.loc[order_df["leadtime_plan_ready"], "date"].min(),
    "~",
    order_df.loc[order_df["leadtime_plan_ready"], "date"].max(),
)

In [ ]:
# ============================================================
# [Quantile 단조성 검증] 후처리 규칙이 항상 적용됐는지 확인
# ============================================================

remaining_crossings = (
    order_df["p95_target_stock_fixed"] < order_df["p90_target_stock"]
).sum()

assert remaining_crossings == 0
assert (order_df["p90_target_stock"] >= 0).all()
assert (order_df["p95_target_stock_fixed"] >= 0).all()

print("수정 전 Test crossing:", int(test_crossing_mask.sum()))
print("수정 후 crossing:", int(remaining_crossings))

## 5. 재고정책 비교

In [ ]:
# ============================================================
# [정적 정책 비교] 리드타임 0일의 당일 확보재고 비교
# ============================================================

Bmart["sales_rolling_mean_7"] = (
    Bmart.groupby(["store_id", "product_id"])["units_sold"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=7).mean())
)

baseline_stock = Bmart.loc[
    Bmart["date"].between(test_df["date"].min(), test_df["date"].max()),
    ["date", "store_id", "product_id", "sales_rolling_mean_7"],
].copy()

order_df = order_df.merge(
    baseline_stock,
    on=["date", "store_id", "product_id"],
    how="left",
    validate="one_to_one",
)

if order_df["sales_rolling_mean_7"].isna().any():
    raise ValueError("Baseline merge 후 결측치가 존재합니다.")

order_df["baseline_order_qty"] = np.ceil(np.maximum(
    order_df["sales_rolling_mean_7"] - order_df["inventory_level"], 0
)).astype(int)

order_df["baseline_available_stock"] = order_df["inventory_level"] + order_df["baseline_order_qty"]
order_df["p90_available_stock"] = order_df["inventory_level"] + order_df["p90_order_qty"]
order_df["p95_available_stock"] = order_df["inventory_level"] + order_df["p95_order_qty_fixed"]


def evaluate_inventory_policy(df, stock_col, order_col, name):
    actual = df["demand_qty"].to_numpy()
    stock = df[stock_col].to_numpy()
    shortage = np.maximum(actual - stock, 0)
    remaining = np.maximum(stock - actual, 0)
    return {
        "Policy": name,
        "Evaluation_Type": "Static, zero lead time",
        "Service_Level": np.mean(stock >= actual) * 100,
        "Stockout_Rate": np.mean(stock < actual) * 100,
        "Fill_Rate": (1 - shortage.sum() / actual.sum()) * 100,
        "Avg_Order_Qty_All_SKUDays": df[order_col].mean(),
        "Total_Order_Qty": df[order_col].sum(),
        "Avg_Remaining": remaining.mean(),
        "Total_Remaining": remaining.sum(),
        "Total_Shortage": shortage.sum(),
    }


static_policy_results = [
    evaluate_inventory_policy(order_df, "baseline_available_stock", "baseline_order_qty", "기존 7일 평균 판매량"),
    evaluate_inventory_policy(order_df, "p90_available_stock", "p90_order_qty", "V2 P90"),
    evaluate_inventory_policy(order_df, "p95_available_stock", "p95_order_qty_fixed", "V2 P95 Fixed"),
]

policy_comparison = pd.DataFrame(static_policy_results)
display(policy_comparison.round(2))
print("주의: 이 표는 동적 재고최적화가 아니라 당일 정적 비교입니다.")

In [ ]:
# ============================================================
# [동적 재고 시뮬레이션]
# 입고예정 pipeline·MOQ·박스 단위를 반영한 order-up-to 정책
# ============================================================

simulation_order_df = order_df[order_df["leadtime_plan_ready"]].copy()
simulation_order_df["baseline_protection_target"] = (
    simulation_order_df["sales_rolling_mean_7"] * PROTECTION_PERIOD_DAYS
)
INITIAL_PIPELINE_COLUMNS = [
    f"initial_incoming_day_{day}" for day in range(1, LEAD_TIME_DAYS + 1)
]


def simulate_base_stock_policy(df, target_col, policy_name):
    records = []

    for (store_id, product_id), group in df.groupby(
        ["store_id", "product_id"], sort=False
    ):
        group = group.sort_values("date")
        first_row = group.iloc[0]
        on_hand = float(first_row["inventory_level"])
        pipeline = [float(first_row[column]) for column in INITIAL_PIPELINE_COLUMNS]

        for _, row in group.iterrows():
            receipt = pipeline.pop(0) if LEAD_TIME_DAYS else 0.0
            on_hand += receipt
            opening_inventory = on_hand

            inventory_position_before_order = (
                on_hand + sum(pipeline) - float(row["backorder_qty"])
            )
            target_position = max(float(row[target_col]), 0.0)
            raw_order_qty = max(
                target_position - inventory_position_before_order,
                0.0,
            )
            order_qty = float(calculate_constrained_order_qty(
                target_position,
                on_hand,
                sum(pipeline),
                row["backorder_qty"],
                row["pack_size"],
                row["minimum_order_qty"],
            ))

            if LEAD_TIME_DAYS:
                pipeline.append(order_qty)
            else:
                on_hand += order_qty

            demand = float(row["demand_qty"])
            fulfilled = min(on_hand, demand)
            shortage = demand - fulfilled
            on_hand -= fulfilled

            records.append({
                "date": row["date"],
                "store_id": store_id,
                "product_id": product_id,
                "policy": policy_name,
                "opening_inventory": opening_inventory,
                "receipt_qty": receipt,
                "target_position": target_position,
                "inventory_position_before_order": inventory_position_before_order,
                "raw_order_qty": raw_order_qty,
                "order_qty": order_qty,
                "constraint_applied": not np.isclose(order_qty, np.ceil(raw_order_qty)),
                "pack_size": float(row["pack_size"]),
                "minimum_order_qty": float(row["minimum_order_qty"]),
                "demand_qty": demand,
                "fulfilled_qty": fulfilled,
                "shortage_qty": shortage,
                "ending_inventory": on_hand,
                "pipeline_after_order": sum(pipeline),
            })

    return pd.DataFrame(records)


dynamic_frames = [
    simulate_base_stock_policy(
        simulation_order_df, "baseline_protection_target", "기존 7일 평균 판매량"
    ),
    simulate_base_stock_policy(
        simulation_order_df, "leadtime_p90_target_stock", "누적수요 V2 P90"
    ),
    simulate_base_stock_policy(
        simulation_order_df, "leadtime_p95_target_stock", "누적수요 V2 P95"
    ),
]
dynamic_simulation_df = pd.concat(dynamic_frames, ignore_index=True)

dynamic_policy_comparison = (
    dynamic_simulation_df.groupby("policy")
    .agg(
        Service_Level=("shortage_qty", lambda x: (x <= 0).mean() * 100),
        Total_Order_Qty=("order_qty", "sum"),
        Order_Cycles=("order_qty", lambda x: x.gt(0).sum()),
        Constraint_Applied_Count=("constraint_applied", "sum"),
        Avg_Ending_Inventory=("ending_inventory", "mean"),
        Total_Shortage=("shortage_qty", "sum"),
        Total_Demand=("demand_qty", "sum"),
    )
    .reset_index()
)
dynamic_policy_comparison["Stockout_Rate"] = (
    100 - dynamic_policy_comparison["Service_Level"]
)
dynamic_policy_comparison["Fill_Rate"] = (
    1 - dynamic_policy_comparison["Total_Shortage"]
    / dynamic_policy_comparison["Total_Demand"]
) * 100
dynamic_policy_comparison["Lead_Time_Days"] = LEAD_TIME_DAYS
dynamic_policy_comparison["Review_Period_Days"] = REVIEW_PERIOD_DAYS
dynamic_policy_comparison["Evaluation_Start"] = str(
    simulation_order_df["date"].min().date()
)
dynamic_policy_comparison["Evaluation_End"] = str(
    simulation_order_df["date"].max().date()
)
dynamic_policy_comparison["Evaluation_Days"] = (
    simulation_order_df["date"].nunique()
)

COST_SCENARIO_ACTIVE = all(
    COST_SCENARIO[key] is not None
    for key in [
        "fixed_order_cost",
        "unit_purchase_cost",
        "unit_holding_cost_per_day",
        "unit_shortage_cost",
    ]
)
if COST_SCENARIO_ACTIVE:
    dynamic_cost_detail_df, dynamic_cost_comparison = evaluate_cost_scenario(
        dynamic_simulation_df,
        COST_SCENARIO,
    )
else:
    dynamic_cost_detail_df = dynamic_simulation_df.copy()
    dynamic_cost_comparison = dynamic_policy_comparison[["policy"]].copy()
    for column in [
        "fixed_order_cost",
        "purchase_cost",
        "holding_cost",
        "shortage_cost",
        "total_cost",
    ]:
        dynamic_cost_comparison[column] = np.nan
    dynamic_cost_comparison["currency"] = None

display(dynamic_policy_comparison.round(2))
print(
    "가정: lost sales 방식. 입고예정 pipeline·MOQ·박스 단위를 지원합니다. "
    "실제 비용이 입력되기 전에는 비용 순위를 계산하지 않습니다. "
    "또한 demand_qty는 검증된 잠재수요가 아니므로 서비스 수준은 시나리오 지표입니다."
)

In [ ]:
# ============================================================
# [V2 셀 25]
# 과잉재고량 계산 + 재고 상태 분류
# ============================================================


# ------------------------------------------------------------
# 1. P95 기준 목표재고
# ------------------------------------------------------------

order_df["target_stock"] = (
    order_df["p95_target_stock_fixed"]
)


# ------------------------------------------------------------
# 2. 부족재고 / 과잉재고 계산
# ------------------------------------------------------------

order_df["shortage_to_target"] = np.maximum(
    order_df["target_stock"]
    -
    order_df["inventory_level"],
    0
)


order_df["excess_inventory"] = np.maximum(
    order_df["inventory_level"]
    -
    order_df["target_stock"],
    0
)


# ------------------------------------------------------------
# 3. 목표재고 대비 현재재고 비율
# ------------------------------------------------------------

order_df["inventory_ratio"] = (
    order_df["inventory_level"]
    /
    order_df["target_stock"]
)


# ------------------------------------------------------------
# 4. 재고 상태 분류 함수
# ------------------------------------------------------------

def classify_inventory_status(row):

    inventory = row["inventory_level"]
    target = row["target_stock"]

    # 목표재고보다 부족
    if inventory < target:
        return "발주 필요"

    # 목표재고의 150% 이상
    elif inventory >= target * 1.5:
        return "과잉재고"

    # 목표재고 이상 ~ 150% 미만
    else:
        return "적정 재고"


order_df["inventory_status"] = (
    order_df.apply(
        classify_inventory_status,
        axis=1
    )
)


# ------------------------------------------------------------
# 5. 상태별 요약
# ------------------------------------------------------------

inventory_status_summary = (
    order_df
    .groupby("inventory_status")
    .agg(
        count=("product_id", "size"),

        avg_inventory=(
            "inventory_level",
            "mean"
        ),

        avg_target=(
            "target_stock",
            "mean"
        ),

        avg_order_qty=(
            "p95_order_qty_fixed",
            "mean"
        ),

        avg_excess_inventory=(
            "excess_inventory",
            "mean"
        )
    )
    .round(2)
)


inventory_status_summary[
    "ratio"
] = (
    inventory_status_summary["count"]
    /
    len(order_df)
    * 100
).round(2)


print(
    "===== 재고 상태 요약 ====="
)

display(
    inventory_status_summary
)


# ------------------------------------------------------------
# 6. 과잉재고 상품 확인
# ------------------------------------------------------------

print(
    "\n===== 과잉재고 상위 상품 ====="
)

display(
    order_df.loc[
        order_df[
            "inventory_status"
        ] == "과잉재고",
        [
            "date",
            "store_id",
            "product_id",
            "category",
            "inventory_level",
            "point_forecast",
            "target_stock",
            "excess_inventory",
            "inventory_ratio",
            "promotion_flag",
            "discount_pct"
        ]
    ]
    .sort_values(
        "excess_inventory",
        ascending=False
    )
    .head(20)
)

In [ ]:
# ============================================================
# [V2 셀 26]
# 발주 우선순위 + 과잉재고 관리 우선순위
# ============================================================


# ------------------------------------------------------------
# 1. 발주 우선순위 점수
#
# 목표재고 대비 부족 비율
# ------------------------------------------------------------

order_df["shortage_ratio"] = (
    order_df["shortage_to_target"]
    /
    order_df["target_stock"]
).fillna(0)


# ------------------------------------------------------------
# 2. 과잉재고 비율
# ------------------------------------------------------------

order_df["excess_ratio"] = (
    order_df["excess_inventory"]
    /
    order_df["target_stock"]
).fillna(0)


# ------------------------------------------------------------
# 3. 발주 위험 등급
# ------------------------------------------------------------

def classify_order_priority(row):

    if row["inventory_status"] != "발주 필요":
        return "해당 없음"

    ratio = row["shortage_ratio"]

    if ratio >= 0.5:
        return "긴급"

    elif ratio >= 0.25:
        return "높음"

    else:
        return "보통"


order_df["order_priority"] = (
    order_df.apply(
        classify_order_priority,
        axis=1
    )
)


# ------------------------------------------------------------
# 4. 과잉재고 관리 등급
# ------------------------------------------------------------

def classify_excess_priority(row):

    if row["inventory_status"] != "과잉재고":
        return "해당 없음"

    ratio = row["inventory_ratio"]

    if ratio >= 3:
        return "심각"

    elif ratio >= 2:
        return "높음"

    else:
        return "주의"


order_df["excess_priority"] = (
    order_df.apply(
        classify_excess_priority,
        axis=1
    )
)


# ------------------------------------------------------------
# 5. 발주 우선순위 요약
# ------------------------------------------------------------

print("===== 발주 우선순위 =====")

display(
    order_df[
        order_df["inventory_status"] == "발주 필요"
    ]
    .groupby("order_priority")
    .agg(
        count=("product_id", "size"),
        avg_inventory=("inventory_level", "mean"),
        avg_target=("target_stock", "mean"),
        avg_order_qty=("p95_order_qty_fixed", "mean")
    )
    .round(2)
)


# ------------------------------------------------------------
# 6. 과잉재고 관리 우선순위 요약
# ------------------------------------------------------------

print("\n===== 과잉재고 관리 우선순위 =====")

display(
    order_df[
        order_df["inventory_status"] == "과잉재고"
    ]
    .groupby("excess_priority")
    .agg(
        count=("product_id", "size"),
        avg_inventory=("inventory_level", "mean"),
        avg_target=("target_stock", "mean"),
        avg_excess=("excess_inventory", "mean")
    )
    .round(2)
)


# ------------------------------------------------------------
# 7. 긴급 발주 상품 TOP 20
# ------------------------------------------------------------

print("\n===== 긴급 발주 TOP 20 =====")

display(
    order_df[
        order_df["inventory_status"] == "발주 필요"
    ]
    .sort_values(
        "shortage_ratio",
        ascending=False
    )[
        [
            "date",
            "store_id",
            "product_id",
            "category",
            "inventory_level",
            "target_stock",
            "p95_order_qty_fixed",
            "shortage_ratio",
            "order_priority"
        ]
    ]
    .head(20)
)


# ------------------------------------------------------------
# 8. 과잉재고 관리 TOP 20
# ------------------------------------------------------------

print("\n===== 과잉재고 관리 TOP 20 =====")

display(
    order_df[
        order_df["inventory_status"] == "과잉재고"
    ]
    .sort_values(
        "excess_ratio",
        ascending=False
    )[
        [
            "date",
            "store_id",
            "product_id",
            "category",
            "inventory_level",
            "target_stock",
            "excess_inventory",
            "inventory_ratio",
            "excess_priority"
        ]
    ]
    .head(20)
)

## 6. 가설 분석 — 인과가 아닌 연관성 분석

In [ ]:
# ============================================================
# [V2 ADD 1] 가설 1: 프로모션은 Demand에 통계적으로 유의한 영향을 주는가?
# ============================================================

import numpy as np
from scipy.stats import ttest_ind, f_oneway

# 1. 프로모션 유무별 Demand 분포 비교
promo_summary = (
    eda_df
    .groupby("promotion_flag")["demand_qty"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
)

print("===== 프로모션 여부별 Demand 요약 =====")
display(promo_summary.round(2))

plt.figure(figsize=(8, 5))
sns.boxplot(data=eda_df, x="promotion_flag", y="demand_qty")
plt.title("프로모션 여부별 Demand 분포")
plt.xlabel("Promotion (0=No, 1=Yes)")
plt.ylabel("Demand")
plt.show()

# 2. 독립표본 t-test
promo_0 = eda_df.loc[eda_df["promotion_flag"] == 0, "demand_qty"]
promo_1 = eda_df.loc[eda_df["promotion_flag"] == 1, "demand_qty"]

promo_ttest = ttest_ind(promo_1, promo_0, equal_var=False)

print("\n===== 프로모션 여부 t-test =====")
print("t-statistic:", round(promo_ttest.statistic, 4))
print("p-value:", promo_ttest.pvalue)

if promo_ttest.pvalue < 0.05:
    print("=> p < 0.05, 프로모션 유무에 따른 Demand 차이는 통계적으로 유의합니다.")
else:
    print("=> p >= 0.05, 통계적으로 유의하다고 보기 어렵습니다.")

# 3. 같은 매장·상품 기준 프로모션 전후 평균 Demand 차이(lift)
promo_pair = (
    eda_df
    .groupby(["store_id", "product_id", "promotion_flag"])["demand_qty"]
    .mean()
    .unstack()
)

promo_pair = promo_pair.dropna(subset=[0, 1]).copy()
promo_pair["lift"] = promo_pair[1] - promo_pair[0]
promo_pair["lift_pct"] = np.where(
    promo_pair[0] != 0,
    promo_pair["lift"] / promo_pair[0] * 100,
    np.nan
)

print("\n===== 매장×상품 단위 프로모션 Lift 상위 20개 =====")
display(promo_pair.sort_values("lift_pct", ascending=False).head(20).round(2))
print("\n프로모션 Lift 평균(%):", round(promo_pair["lift_pct"].mean(), 2))
print("주의: 반복 시계열 관측과 할인율 교란이 있으므로 연관성으로만 해석합니다.")

In [ ]:
# ============================================================
# [V2 ADD 2] 가설 2: 할인율이 높을수록 Demand가 증가하는가?
# ============================================================

# 1. 할인율 - Demand 상관계수
discount_corr = eda_df[["discount_pct", "demand_qty"]].corr().iloc[0, 1]
print("===== 할인율 ↔ Demand 상관계수 =====")
print(round(discount_corr, 4))

# 2. 할인율 구간별 평균 Demand 추세
discount_summary = (
    eda_df
    .groupby("discount_pct")["demand_qty"]
    .agg(["mean", "median", "std", "count"])
    .reset_index()
)

print("\n===== 할인율별 Demand 요약 =====")
display(discount_summary.round(2))

plt.figure(figsize=(9, 5))
sns.lineplot(data=discount_summary, x="discount_pct", y="mean", marker="o")
plt.title("할인율별 평균 Demand 추세")
plt.xlabel("Discount %")
plt.ylabel("Mean Demand")
plt.show()

# 3. 할인율 구간 간 ANOVA
discount_groups = [g["demand_qty"].values for _, g in eda_df.groupby("discount_pct")]
discount_anova = f_oneway(*discount_groups)

print("\n===== 할인율 구간 ANOVA =====")
print("F-statistic:", round(discount_anova.statistic, 4))
print("p-value:", discount_anova.pvalue)

if discount_anova.pvalue < 0.05:
    print("=> p < 0.05, 할인율 구간에 따른 Demand 차이는 통계적으로 유의합니다.")
else:
    print("=> p >= 0.05, 통계적으로 유의하다고 보기 어렵습니다.")
print("주의: ANOVA는 단조 증가를 검정하지 않으며 그룹 간 차이만 검정합니다.")

In [ ]:
# ============================================================
# [V2 셀 27]
# 가설 3: 지역 및 계절에 따라
# 주력 상품 카테고리가 달라지는가?
# ============================================================

region_season_category = (
    eda_df
    .groupby([
        "region",
        "seasonality",
        "category"
    ])
    .agg(
        avg_demand=("demand_qty", "mean"),
        total_demand=("demand_qty", "sum"),
        count=("demand_qty", "size")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 각 지역 + 계절별 평균 Demand 1위 카테고리
# ------------------------------------------------------------

top_category = (
    region_season_category
    .sort_values(
        [
            "region",
            "seasonality",
            "avg_demand"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .groupby(
        [
            "region",
            "seasonality"
        ]
    )
    .head(1)
    .reset_index(drop=True)
)


print(
    "===== 지역·계절별 평균 Demand 1위 카테고리 ====="
)

display(
    top_category[
        [
            "region",
            "seasonality",
            "category",
            "avg_demand",
            "total_demand"
        ]
    ].round(2)
)


# ------------------------------------------------------------
# 지역별 1위 카테고리 종류 확인
# ------------------------------------------------------------

print(
    "\n===== 지역별 주력 카테고리 종류 수 ====="
)

display(
    top_category
    .groupby("region")["category"]
    .nunique()
    .reset_index(
        name="top_category_nunique"
    )
)

In [ ]:
# ============================================================
# [V2 셀 ADD 3] 가설 3 증명 보완
# ============================================================

season_summary = (
    eda_df.groupby("seasonality")["demand_qty"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

store_summary = (
    eda_df.groupby("store_id")["demand_qty"]
    .agg(["mean", "median", "std", "count"])
    .sort_values("mean", ascending=False)
)

print("===== 계절별 Demand 요약 =====")
display(season_summary.round(2))
print("\n===== 매장별 Demand 요약 =====")
display(store_summary.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=eda_df, x="seasonality", y="demand_qty", estimator="mean", errorbar=None, ax=axes[0])
axes[0].set_title("계절별 평균 Demand")

sns.barplot(data=eda_df, x="store_id", y="demand_qty", estimator="mean", errorbar=None, ax=axes[1])
axes[1].set_title("매장별 평균 Demand")

plt.tight_layout()
plt.show()

# 그룹 간 차이에 대한 ANOVA
season_groups = [g["demand_qty"].values for _, g in eda_df.groupby("seasonality")]
store_groups = [g["demand_qty"].values for _, g in eda_df.groupby("store_id")]

season_anova = f_oneway(*season_groups)
store_anova = f_oneway(*store_groups)

print("===== 계절 ANOVA =====")
print("F-statistic:", round(season_anova.statistic, 4))
print(f"p-value: {season_anova.pvalue:.10f}")

print("\n===== 매장 ANOVA =====")
print("F-statistic:", round(store_anova.statistic, 4))
print(f"p-value: {store_anova.pvalue:.10f}")

In [ ]:
# ============================================================
# [가설 4] 재고 소진 의심일과 다음 실제 날짜의 Demand 관계
# ============================================================

hypothesis_df = eda_df.sort_values(
    ["store_id", "product_id", "date"]
).copy()
hypothesis_df["stockout_candidate"] = (
    (hypothesis_df["inventory_level"] > 0)
    & (hypothesis_df["units_sold"] >= hypothesis_df["inventory_level"])
)
hypothesis_df["next_date"] = hypothesis_df.groupby(
    ["store_id", "product_id"]
)["date"].shift(-1)
hypothesis_df["next_day_demand"] = hypothesis_df.groupby(
    ["store_id", "product_id"]
)["demand_qty"].shift(-1)
hypothesis_df = hypothesis_df[
    hypothesis_df["next_date"] - hypothesis_df["date"] == pd.Timedelta(days=1)
].copy()
hypothesis_df["next_demand_change"] = (
    hypothesis_df["next_day_demand"] - hypothesis_df["demand_qty"]
)

stockout_next_demand = (
    hypothesis_df.groupby("stockout_candidate")
    .agg(
        count=("next_day_demand", "size"),
        current_avg_demand=("demand_qty", "mean"),
        next_avg_demand=("next_day_demand", "mean"),
        avg_change=("next_demand_change", "mean"),
        median_change=("next_demand_change", "median"),
    )
    .round(2)
)
display(stockout_next_demand)
print("주의: 결품 라벨이 아닌 재고 소진 의심 기준이며 인과효과로 해석하지 않습니다.")

## 7. Streamlit용 결과와 Artifact 저장

In [ ]:
# ============================================================
# [V2 셀 29]
# Streamlit / 서비스용 최종 결과 테이블
# ============================================================

service_df = order_df[
    [
        "date",
        "store_id",
        "product_id",
        "category",
        "region",

        "inventory_level",
        "incoming_order_qty",
        "backorder_qty",
        "inventory_position",
        "pack_size",
        "minimum_order_qty",
        "demand_qty",

        "point_forecast",
        "p90_target_stock",
        "p95_target_stock_fixed",

        "p90_order_qty",
        "p95_order_qty_unconstrained",
        "p95_order_qty_fixed",
        "order_constraint_applied",

        "inventory_status",

        "order_priority",
        "excess_priority",

        "excess_inventory",
        "shortage_to_target",

        "promotion_flag",
        "discount_pct",
        "price"
    ]
].copy()


# ------------------------------------------------------------
# 컬럼명 조금 더 보기 쉽게 변경
# ------------------------------------------------------------

service_df = service_df.rename(columns={

    "p95_target_stock_fixed":
        "recommended_target_stock",

    "p95_order_qty_fixed":
        "recommended_order_qty"
})


# ------------------------------------------------------------
# 위험도 통합 컬럼
# ------------------------------------------------------------

def make_management_action(row):

    if row["inventory_status"] == "발주 필요":

        if row["order_priority"] == "긴급":
            return "긴급 발주"

        elif row["order_priority"] == "높음":
            return "우선 발주"

        else:
            return "발주 검토"


    elif row["inventory_status"] == "과잉재고":

        if row["excess_priority"] == "심각":
            return "과잉재고 긴급 관리"

        elif row["excess_priority"] == "높음":
            return "과잉재고 관리"

        else:
            return "과잉재고 주의"


    else:
        return "정상 유지"


service_df["management_action"] = (
    service_df.apply(
        make_management_action,
        axis=1
    )
)


# ------------------------------------------------------------
# 정렬
# ------------------------------------------------------------

service_df = (
    service_df
    .sort_values(
        [
            "date",
            "store_id",
            "product_id"
        ]
    )
    .reset_index(drop=True)
)


print("===== 서비스용 최종 데이터 =====")

print(
    "행 수:",
    len(service_df)
)

print(
    "컬럼 수:",
    len(service_df.columns)
)


display(
    service_df.head(20)
)


print("\n===== 관리 상태 분포 =====")

display(
    service_df[
        "management_action"
    ]
    .value_counts()
)

In [ ]:
# ============================================================
# [Streamlit 공통 추론 함수] 1일 선행 예측
# ============================================================

FORECAST_REQUIRED_COLUMNS = (set(base_features) - {"day_of_week"}) | {
    "date", "inventory_level", "is_holiday"
}


def build_one_day_features(history_df, forecast_rows):
    history = history_df.copy()
    forecast = ensure_order_inputs(forecast_rows)

    history["date"] = pd.to_datetime(history["date"], errors="raise")
    forecast["date"] = pd.to_datetime(forecast["date"], errors="raise")

    missing_forecast_columns = FORECAST_REQUIRED_COLUMNS - set(forecast.columns)
    if missing_forecast_columns:
        raise ValueError(
            f"미래 입력 필수 컬럼 누락: {sorted(missing_forecast_columns)}"
        )

    history_required = {"date", "store_id", "product_id", "demand_qty"}
    missing_history_columns = history_required - set(history.columns)
    if missing_history_columns:
        raise ValueError(
            f"수요 이력 필수 컬럼 누락: {sorted(missing_history_columns)}"
        )

    if forecast.duplicated(["date", "store_id", "product_id"]).any():
        raise ValueError("미래 입력의 date-store-product 키가 중복되었습니다.")

    latest_history = (
        history.groupby(["store_id", "product_id"])["date"]
        .max()
        .rename("latest_history_date")
        .reset_index()
    )
    forecast = forecast.merge(
        latest_history,
        on=["store_id", "product_id"],
        how="left",
        validate="many_to_one",
    )
    forecast["is_next_day"] = (
        forecast["date"] - forecast["latest_history_date"]
        == pd.Timedelta(days=1)
    )

    history_window_status = []
    for (store_id, product_id), group in history.groupby(
        ["store_id", "product_id"], sort=False
    ):
        recent_dates = group.sort_values("date")["date"].tail(28)
        consecutive = (
            len(recent_dates) == 28
            and recent_dates.diff().dropna().eq(pd.Timedelta(days=1)).all()
        )
        history_window_status.append({
            "store_id": store_id,
            "product_id": product_id,
            "consecutive_28_day_history": bool(consecutive),
        })

    forecast = forecast.merge(
        pd.DataFrame(history_window_status),
        on=["store_id", "product_id"],
        how="left",
        validate="many_to_one",
    )

    history_small = history[["date", "store_id", "product_id", "demand_qty"]].copy()
    history_small["_forecast_row"] = False
    forecast["demand_qty"] = np.nan
    forecast["_forecast_row"] = True

    combined = pd.concat([history_small, forecast], ignore_index=True, sort=False)
    combined = combined.sort_values(
        ["store_id", "product_id", "date", "_forecast_row"]
    ).reset_index(drop=True)

    for lag in [1, 7, 14, 28]:
        combined[f"demand_lag_{lag}"] = (
            combined.groupby(["store_id", "product_id"])["demand_qty"]
            .shift(lag)
        )

    combined["demand_rolling_mean_7"] = (
        combined.groupby(["store_id", "product_id"])["demand_qty"]
        .transform(lambda x: x.shift(1).rolling(7, min_periods=7).mean())
    )
    combined["demand_rolling_mean_28"] = (
        combined.groupby(["store_id", "product_id"])["demand_qty"]
        .transform(lambda x: x.shift(1).rolling(28, min_periods=28).mean())
    )
    combined["demand_rolling_std_7"] = (
        combined.groupby(["store_id", "product_id"])["demand_qty"]
        .transform(lambda x: x.shift(1).rolling(7, min_periods=7).std())
    )

    prediction_features = combined[combined["_forecast_row"]].copy()
    prediction_features["day_of_week"] = prediction_features["date"].dt.day_name()
    prediction_features["year"] = prediction_features["date"].dt.year
    prediction_features["month"] = prediction_features["date"].dt.month
    prediction_features["day"] = prediction_features["date"].dt.day
    prediction_features["month_sin"] = np.sin(
        2 * np.pi * prediction_features["month"] / 12
    )
    prediction_features["month_cos"] = np.cos(
        2 * np.pi * prediction_features["month"] / 12
    )

    prediction_features["history_ready"] = (
        prediction_features[time_features].notna().all(axis=1)
        & prediction_features["is_next_day"].fillna(False)
        & prediction_features["consecutive_28_day_history"].fillna(False)
    )
    return prediction_features


def predict_one_day_inventory(history_df, forecast_rows):
    prepared = build_one_day_features(history_df, forecast_rows)
    result = prepared[
        [
            "date", "store_id", "product_id", "inventory_level",
            "incoming_order_qty", "backorder_qty", "pack_size",
            "minimum_order_qty", "history_ready",
        ]
    ].copy()
    result["prediction_status"] = np.where(
        result["history_ready"], "ok", "insufficient_or_nonconsecutive_history"
    )

    ready_mask = prepared["history_ready"]
    if ready_mask.any():
        ready_features = prepared.loc[ready_mask, feature_columns]
        point = np.maximum(final_point_model.predict(ready_features), 0)
        p90 = np.maximum(
            final_p90_model.predict(ready_features) + P90_CORRECTION, 0
        )
        p95_raw = np.maximum(
            final_p95_model.predict(ready_features) + P95_CORRECTION, 0
        )
        p95 = np.maximum(p95_raw, p90)
        ready_rows = prepared.loc[ready_mask]
        inventory_position = (
            ready_rows["inventory_level"]
            + ready_rows["incoming_order_qty"]
            - ready_rows["backorder_qty"]
        )

        result.loc[ready_mask, "inventory_position"] = inventory_position
        result.loc[ready_mask, "point_forecast"] = point
        result.loc[ready_mask, "p90_target_stock"] = p90
        result.loc[ready_mask, "recommended_target_stock"] = p95
        result.loc[ready_mask, "unconstrained_order_qty"] = np.ceil(
            np.maximum(p95 - inventory_position.to_numpy(), 0)
        ).astype(int)
        result.loc[ready_mask, "recommended_order_qty"] = (
            calculate_constrained_order_qty(
                p95,
                ready_rows["inventory_level"],
                ready_rows["incoming_order_qty"],
                ready_rows["backorder_qty"],
                ready_rows["pack_size"],
                ready_rows["minimum_order_qty"],
            )
        )

    return result.sort_values(
        ["date", "store_id", "product_id"]
    ).reset_index(drop=True)



PROTECTION_PLAN_REQUIRED_COLUMNS = {
    "date",
    "store_id",
    "product_id",
    *LEADTIME_REQUIRED_PLAN_COLUMNS,
}


def build_protection_period_features(history_df, forecast_plan_rows):
    history = history_df.copy()
    plan = ensure_order_inputs(forecast_plan_rows)
    history["date"] = pd.to_datetime(history["date"], errors="raise")
    plan["date"] = pd.to_datetime(plan["date"], errors="raise")

    missing_plan_columns = PROTECTION_PLAN_REQUIRED_COLUMNS - set(plan.columns)
    if missing_plan_columns:
        raise ValueError(
            f"5일 계획 입력 필수 컬럼 누락: {sorted(missing_plan_columns)}"
        )
    if plan.duplicated(["date", "store_id", "product_id"]).any():
        raise ValueError("5일 계획 입력의 date-store-product 키가 중복되었습니다.")

    latest_history = (
        history.groupby(["store_id", "product_id"])["date"]
        .max()
        .rename("latest_history_date")
    )
    invalid_windows = []
    for key, group in plan.groupby(["store_id", "product_id"], sort=False):
        if key not in latest_history.index:
            invalid_windows.append((*key, "history_missing"))
            continue
        expected_dates = list(pd.date_range(
            latest_history.loc[key] + pd.Timedelta(days=1),
            periods=PROTECTION_PERIOD_DAYS,
            freq="D",
        ))
        observed_dates = list(group["date"].sort_values())
        if observed_dates != expected_dates:
            invalid_windows.append((*key, "five_consecutive_days_required"))

    if invalid_windows:
        raise ValueError(
            "각 매장×상품에 최신 이력 다음 날부터 연속 5일 계획이 필요합니다. "
            f"오류 예시: {invalid_windows[:5]}"
        )

    first_day_rows = (
        plan.sort_values(["store_id", "product_id", "date"])
        .groupby(["store_id", "product_id"], as_index=False)
        .head(1)
    )
    prepared = build_one_day_features(history, first_day_rows)

    plan_aggregates = (
        plan.groupby(["store_id", "product_id"], as_index=False)
        .agg(
            protection_price_mean=("price", "mean"),
            protection_discount_mean=("discount_pct", "mean"),
            protection_promotion_days=("promotion_flag", "sum"),
            protection_holiday_days=("is_holiday", "sum"),
            protection_end_date=("date", "max"),
        )
    )
    prepared = prepared.merge(
        plan_aggregates,
        on=["store_id", "product_id"],
        how="left",
        validate="one_to_one",
    )
    prepared["protection_history_ready"] = (
        prepared["history_ready"]
        & prepared[LEADTIME_FUTURE_FEATURES].notna().all(axis=1)
    )
    return prepared


def predict_protection_period_inventory(history_df, forecast_plan_rows):
    prepared = build_protection_period_features(history_df, forecast_plan_rows)
    result = prepared[[
        "date",
        "protection_end_date",
        "store_id",
        "product_id",
        "inventory_level",
        "incoming_order_qty",
        "backorder_qty",
        "pack_size",
        "minimum_order_qty",
        "protection_history_ready",
    ]].copy()
    result["prediction_status"] = np.where(
        result["protection_history_ready"],
        "ok",
        "insufficient_or_nonconsecutive_history",
    )

    ready_mask = prepared["protection_history_ready"]
    if ready_mask.any():
        ready_features = prepared.loc[ready_mask, leadtime_feature_columns]
        ready_rows = prepared.loc[ready_mask]
        rolling_corrections = get_rolling_leadtime_corrections(
            ready_rows["date"]
        )
        p90 = np.maximum(
            leadtime_p90_model.predict(ready_features)
            + rolling_corrections["p90_correction"].to_numpy(),
            0,
        )
        p95_raw = np.maximum(
            leadtime_p95_model.predict(ready_features)
            + rolling_corrections["p95_correction"].to_numpy(),
            0,
        )
        p95 = np.maximum(p95_raw, p90)
        inventory_position = (
            ready_rows["inventory_level"]
            + ready_rows["incoming_order_qty"]
            - ready_rows["backorder_qty"]
        )

        result.loc[ready_mask, "inventory_position"] = inventory_position
        result.loc[ready_mask, "protection_p90_target_stock"] = p90
        result.loc[ready_mask, "protection_p95_target_stock"] = p95
        result.loc[ready_mask, "unconstrained_order_qty"] = np.ceil(
            np.maximum(p95 - inventory_position.to_numpy(), 0)
        ).astype(int)
        result.loc[ready_mask, "recommended_order_qty"] = (
            calculate_constrained_order_qty(
                p95,
                ready_rows["inventory_level"],
                ready_rows["incoming_order_qty"],
                ready_rows["backorder_qty"],
                ready_rows["pack_size"],
                ready_rows["minimum_order_qty"],
            )
        )

    return result.sort_values(
        ["date", "store_id", "product_id"]
    ).reset_index(drop=True)


print("Streamlit 1일·5일 추론 및 발주 제약 함수 준비 완료")

In [ ]:
# ============================================================
# [운영 모니터링·재학습 기준]
# 단일 일자보다 반복되는 경고를 기준으로 재학습을 판단한다.
# ============================================================

MONITORING_THRESHOLDS = {
    "point_wape_warning_pct": 6.0,
    "point_wape_retrain_pct": 8.0,
    "absolute_bias_warning_pct": 3.0,
    "p90_coverage_min_pct": 87.0,
    "p90_coverage_max_pct": 93.0,
    "p95_coverage_min_pct": 92.0,
    "p95_coverage_max_pct": 98.0,
    "psi_warning": 0.20,
    "psi_retrain": 0.30,
    "missing_feature_rate_retrain_pct": 0.0,
    "unknown_category_rate_warning_pct": 1.0,
    "consecutive_warning_windows_for_retrain": 3,
}
RETRAIN_POLICY = {
    "evaluation_frequency": "weekly after actual demand is available",
    "hard_triggers": [
        "required schema mismatch",
        "required feature missing rate above threshold",
        "quantile crossing after post-processing",
    ],
    "persistent_trigger": (
        "warning status for 3 consecutive weekly windows"
    ),
    "minimum_actions_before_retrain": [
        "confirm input pipeline and future-plan quality",
        "compare against latest rolling baseline",
        "recalibrate quantiles before full model retraining",
    ],
}


def population_stability_index(reference, current, bins=10):
    reference = np.asarray(reference, dtype=float)
    current = np.asarray(current, dtype=float)
    reference = reference[np.isfinite(reference)]
    current = current[np.isfinite(current)]
    if reference.size == 0 or current.size == 0:
        return np.nan
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if edges.size < 3:
        return 0.0 if np.isclose(reference.mean(), current.mean()) else np.inf
    edges[0] = -np.inf
    edges[-1] = np.inf
    reference_count, _ = np.histogram(reference, bins=edges)
    current_count, _ = np.histogram(current, bins=edges)
    epsilon = 1e-6
    reference_pct = np.clip(reference_count / reference_count.sum(), epsilon, None)
    current_pct = np.clip(current_count / current_count.sum(), epsilon, None)
    return float(np.sum(
        (current_pct - reference_pct)
        * np.log(current_pct / reference_pct)
    ))


def build_prediction_monitoring_metrics(actual, point, p90, p95):
    actual = np.asarray(actual, dtype=float)
    point = np.asarray(point, dtype=float)
    p90 = np.asarray(p90, dtype=float)
    p95 = np.asarray(p95, dtype=float)
    if not (actual.shape == point.shape == p90.shape == p95.shape):
        raise ValueError("모니터링 입력 길이가 서로 다릅니다.")
    denominator = np.abs(actual).sum()
    error = point - actual
    return {
        "Rows": len(actual),
        "Point_WAPE_Pct": float(
            np.abs(error).sum() / denominator * 100
        ) if denominator else np.nan,
        "Point_Bias_Pct": float(
            error.sum() / denominator * 100
        ) if denominator else np.nan,
        "P90_Coverage_Pct": float((actual <= p90).mean() * 100),
        "P95_Coverage_Pct": float((actual <= p95).mean() * 100),
        "Crossing_Rate_Pct": float((p90 > p95).mean() * 100),
    }


def monitoring_status(metric, value):
    if metric == "Point_WAPE_Pct":
        if value >= MONITORING_THRESHOLDS["point_wape_retrain_pct"]:
            return "retrain"
        if value >= MONITORING_THRESHOLDS["point_wape_warning_pct"]:
            return "warning"
        return "ok"
    if metric == "Absolute_Bias_Pct":
        return (
            "warning"
            if value >= MONITORING_THRESHOLDS["absolute_bias_warning_pct"]
            else "ok"
        )
    if metric == "P90_Coverage_Pct":
        low = MONITORING_THRESHOLDS["p90_coverage_min_pct"]
        high = MONITORING_THRESHOLDS["p90_coverage_max_pct"]
        return "ok" if low <= value <= high else "warning"
    if metric == "P95_Coverage_Pct":
        low = MONITORING_THRESHOLDS["p95_coverage_min_pct"]
        high = MONITORING_THRESHOLDS["p95_coverage_max_pct"]
        return "ok" if low <= value <= high else "warning"
    if metric == "Crossing_Rate_Pct":
        return "ok" if value == 0 else "retrain"
    return "not_evaluated"


test_monitoring_metrics = build_prediction_monitoring_metrics(
    y_test,
    test_point_pred,
    test_p90_calibrated,
    test_p95_calibrated,
)
monitoring_rows = [
    {"Scope": "daily_test", "Metric": key, "Value": value}
    for key, value in test_monitoring_metrics.items()
    if key != "Rows"
]
monitoring_rows.extend([
    {
        "Scope": "daily_test",
        "Metric": "Absolute_Bias_Pct",
        "Value": abs(test_monitoring_metrics["Point_Bias_Pct"]),
    },
    {
        "Scope": "protection_period_test",
        "Metric": "P90_Coverage_Pct",
        "Value": float(leadtime_model_evaluation.loc[0, "P90_Coverage"]),
    },
    {
        "Scope": "protection_period_test",
        "Metric": "P95_Coverage_Pct",
        "Value": float(leadtime_model_evaluation.loc[0, "P95_Coverage"]),
    },
])
monitoring_baseline_df = pd.DataFrame(monitoring_rows)
monitoring_baseline_df["Status"] = monitoring_baseline_df.apply(
    lambda row: monitoring_status(row["Metric"], row["Value"]),
    axis=1,
)

if test_monitoring_metrics["Crossing_Rate_Pct"] != 0:
    raise ValueError("후처리 후 Quantile crossing이 남아 있습니다.")

display(monitoring_baseline_df.round(4))
print(
    "재학습 판단: 입력 파이프라인 확인 → Quantile 재보정 → "
    "3주 연속 경고 또는 hard trigger일 때 전체 재학습"
)

In [ ]:
# ============================================================
# [Artifact 저장] 모델·정적/동적 정책·검증·메타데이터
# ============================================================

final_point_model.save_model("Bmart_v2_point_model.cbm")
final_p90_model.save_model("Bmart_v2_quantile_p90.cbm")
final_p95_model.save_model("Bmart_v2_quantile_p95.cbm")
leadtime_p90_model.save_model("Bmart_v2_leadtime_quantile_p90.cbm")
leadtime_p95_model.save_model("Bmart_v2_leadtime_quantile_p95.cbm")

service_df.to_csv("Bmart_v2_service_result.csv", index=False, encoding="utf-8-sig")
policy_comparison.to_csv("Bmart_v2_inventory_policy_comparison.csv", index=False, encoding="utf-8-sig")
dynamic_policy_comparison.to_csv("Bmart_v2_dynamic_policy_comparison.csv", index=False, encoding="utf-8-sig")
dynamic_simulation_df.to_csv("Bmart_v2_dynamic_simulation_detail.csv", index=False, encoding="utf-8-sig")
dynamic_cost_comparison.to_csv("Bmart_v2_dynamic_cost_comparison.csv", index=False, encoding="utf-8-sig")
rolling_result_df.to_csv("Bmart_v2_rolling_validation.csv", index=False, encoding="utf-8-sig")
quantile_rolling_result_df.to_csv("Bmart_v2_quantile_rolling_validation.csv", index=False, encoding="utf-8-sig")
point_series_performance_df.to_csv("Bmart_v2_point_series_performance.csv", index=False, encoding="utf-8-sig")
quantile_slice_coverage_df.to_csv("Bmart_v2_quantile_slice_coverage.csv", index=False, encoding="utf-8-sig")
leadtime_model_evaluation.to_csv("Bmart_v2_leadtime_model_evaluation.csv", index=False, encoding="utf-8-sig")
leadtime_calibration_selection.to_csv("Bmart_v2_leadtime_calibration_selection.csv", index=False, encoding="utf-8-sig")
leadtime_residual_history.to_csv("Bmart_v2_leadtime_calibration_residuals.csv", index=False, encoding="utf-8-sig")
target_audit_df.to_csv("Bmart_v2_target_audit.csv", index=False, encoding="utf-8-sig")
monitoring_baseline_df.to_csv("Bmart_v2_monitoring_baseline.csv", index=False, encoding="utf-8-sig")

model_config = {
    "artifact_schema_version": "1.1.0",
    "target": target,
    "target_contract": TARGET_CONTRACT,
    "features": feature_columns,
    "categorical_features": categorical_features,
    "deployment": {
        "inference_module": "bmart_v2_app/inference.py",
        "streamlit_entrypoint": "bmart_v2_app/app.py",
        "artifact_dir_env_var": "BMART_ARTIFACT_DIR",
        "data_path_env_var": "BMART_DATA_PATH",
        "accepts_original_or_normalized_columns": True,
    },
    "forecast_contract": {
        "horizon_days": FORECAST_HORIZON_DAYS,
        "mode": "daily rolling one-step-ahead",
        "requires_latest_actual_demand_history_days": 28,
        "planned_or_forecast_features": planned_or_forecast_features,
    },
    "point_model_params": final_point_model.get_all_params(),
    "p90_model_params": final_p90_model.get_all_params(),
    "p95_model_params": final_p95_model.get_all_params(),
    "random_seed": RANDOM_STATE,
    "p90_correction": P90_CORRECTION,
    "p95_correction": P95_CORRECTION,
    "postprocessing_order": [
        "clip raw quantile predictions at zero",
        "add calibration corrections",
        "P95 = max(P95, P90)",
        "inventory position = on-hand + incoming orders - backorders",
        "raw order = max(target - inventory position, 0)",
        "round up to pack size and then enforce MOQ",
    ],
    "inventory_policy": "P95 Fixed",
    "order_constraints": {
        "defaults": ORDER_INPUT_DEFAULTS,
        "units_ordered_reused_as_incoming": False,
        "cost_scenario": COST_SCENARIO,
        "cost_scenario_active": COST_SCENARIO_ACTIVE,
    },
    "leadtime_demand_model": {
        "target": LEADTIME_TARGET,
        "protection_period_days": PROTECTION_PERIOD_DAYS,
        "feature_columns": leadtime_feature_columns,
        "planned_feature_contract": {
            "known_at_order_time": True,
            "required_daily_columns": LEADTIME_REQUIRED_PLAN_COLUMNS,
            "aggregation_rules": LEADTIME_AGGREGATION_RULES,
        },
        "training_end": str(train_end.date()),
        "calibration_end": str(valid_end.date()),
        "p90_correction": LEADTIME_P90_CORRECTION,
        "p95_correction": LEADTIME_P95_CORRECTION,
        "calibration": {
            "default_method": "rolling_conformal",
            "selection_data": "validation_only",
            "validation_seed_end": str(LEADTIME_VALIDATION_SEED_END.date()),
            "selected_window_days": LEADTIME_CALIBRATION_WINDOW_DAYS,
            "candidate_windows_days": LEADTIME_WINDOW_CANDIDATES[:-1] + ["expanding"],
            "residual_history_file": "Bmart_v2_leadtime_calibration_residuals.csv",
            "refresh_after_target_maturity_days": PROTECTION_PERIOD_DAYS - 1,
            "stale_warning_days": 14,
            "fallback_method": "fixed_full_validation",
        },
        "p90_model_params": leadtime_p90_model.get_all_params(),
        "p95_model_params": leadtime_p95_model.get_all_params(),
    },
    "static_policy_assumption": "same-day receipt / zero lead time",
    "dynamic_simulation_assumptions": {
        "lead_time_days": LEAD_TIME_DAYS,
        "review_period_days": REVIEW_PERIOD_DAYS,
        "initial_pipeline_columns": INITIAL_PIPELINE_COLUMNS,
        "pack_size_and_moq_applied": True,
        "target_method": "direct quantile forecast of cumulative demand over the protection period",
    },
    "data": {
        "source_filename": DATA_PATH.name,
        "path_env_var": "BMART_DATA_PATH",
        "sha256": DATA_SHA256,
        "train_end": str(train_end.date()),
        "validation_end": str(valid_end.date()),
        "test_start": str(test_df["date"].min().date()),
        "test_end": str(test_df["date"].max().date()),
    },
    "monitoring": {
        "thresholds": MONITORING_THRESHOLDS,
        "retrain_policy": RETRAIN_POLICY,
        "baseline_file": "Bmart_v2_monitoring_baseline.csv",
    },
    "environment": {
        "python": platform.python_version(),
        "catboost": catboost.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
    },
    "limitations": [
        "Demand is a dataset-provided target and is not proven latent demand.",
        "Inventory timing and Units Ordered receipt timing are unknown.",
        "Service-level outputs are scenario metrics, not verified operational KPIs.",
        "Static policy metrics are not a dynamic inventory optimization result.",
        "Cost ranking is disabled until real cost inputs are supplied.",
    ],
}

with open("Bmart_v2_model_config.json", "w", encoding="utf-8") as handle:
    json.dump(model_config, handle, ensure_ascii=False, indent=2)

manifest_files = [
    "Bmart_v2_point_model.cbm",
    "Bmart_v2_quantile_p90.cbm",
    "Bmart_v2_quantile_p95.cbm",
    "Bmart_v2_leadtime_quantile_p90.cbm",
    "Bmart_v2_leadtime_quantile_p95.cbm",
    "Bmart_v2_model_config.json",
    "Bmart_v2_leadtime_calibration_residuals.csv",
]
artifact_manifest = {
    "artifact_schema_version": "1.1.0",
    "files": {
        filename: {
            "sha256": hashlib.sha256(Path(filename).read_bytes()).hexdigest(),
            "size_bytes": Path(filename).stat().st_size,
        }
        for filename in manifest_files
    },
}
with open("Bmart_v2_artifact_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(artifact_manifest, handle, ensure_ascii=False, indent=2)

print("Artifact 저장 완료")

In [ ]:
# ============================================================
# [Artifact 검증] 존재 여부뿐 아니라 로드·schema·예측 일치 확인
# ============================================================

saved_files = [
    "Bmart_v2_point_model.cbm",
    "Bmart_v2_quantile_p90.cbm",
    "Bmart_v2_quantile_p95.cbm",
    "Bmart_v2_leadtime_quantile_p90.cbm",
    "Bmart_v2_leadtime_quantile_p95.cbm",
    "Bmart_v2_service_result.csv",
    "Bmart_v2_inventory_policy_comparison.csv",
    "Bmart_v2_dynamic_policy_comparison.csv",
    "Bmart_v2_dynamic_simulation_detail.csv",
    "Bmart_v2_dynamic_cost_comparison.csv",
    "Bmart_v2_rolling_validation.csv",
    "Bmart_v2_quantile_rolling_validation.csv",
    "Bmart_v2_point_series_performance.csv",
    "Bmart_v2_quantile_slice_coverage.csv",
    "Bmart_v2_leadtime_model_evaluation.csv",
    "Bmart_v2_leadtime_calibration_selection.csv",
    "Bmart_v2_leadtime_calibration_residuals.csv",
    "Bmart_v2_target_audit.csv",
    "Bmart_v2_monitoring_baseline.csv",
    "Bmart_v2_model_config.json",
    "Bmart_v2_artifact_manifest.json",
]

missing_files = [file for file in saved_files if not Path(file).exists()]
if missing_files:
    raise FileNotFoundError(f"저장 누락: {missing_files}")

check_service = pd.read_csv("Bmart_v2_service_result.csv")
required_service_columns = {
    "date", "store_id", "product_id", "inventory_level",
    "point_forecast", "recommended_target_stock",
    "recommended_order_qty", "inventory_position", "pack_size",
    "minimum_order_qty", "order_constraint_applied",
    "inventory_status", "management_action",
}
missing_service_columns = required_service_columns - set(check_service.columns)
if missing_service_columns:
    raise ValueError(f"서비스 CSV 필수 컬럼 누락: {sorted(missing_service_columns)}")

if check_service.duplicated(["date", "store_id", "product_id"]).any():
    raise ValueError("서비스 CSV 키 중복 발견")

positive_order = check_service["recommended_order_qty"] > 0
if not (
    check_service.loc[positive_order, "recommended_order_qty"]
    % check_service.loc[positive_order, "pack_size"]
    == 0
).all():
    raise ValueError("추천 발주량이 박스 단위의 배수가 아닙니다.")
if not (
    check_service.loc[positive_order, "recommended_order_qty"]
    >= check_service.loc[positive_order, "minimum_order_qty"]
).all():
    raise ValueError("추천 발주량이 MOQ보다 작습니다.")

loaded_point_model = CatBoostRegressor()
loaded_point_model.load_model("Bmart_v2_point_model.cbm")
loaded_sample_pred = np.maximum(loaded_point_model.predict(X_test.head(100)), 0)
if not np.allclose(loaded_sample_pred, test_point_pred[:100]):
    raise ValueError("저장 후 Point 모델 예측이 원본 예측과 일치하지 않습니다.")

loaded_leadtime_model = CatBoostRegressor()
loaded_leadtime_model.load_model("Bmart_v2_leadtime_quantile_p95.cbm")
leadtime_verify_features = leadtime_model_df.loc[
    X_test.index[:100], leadtime_feature_columns
]
loaded_leadtime_pred = loaded_leadtime_model.predict(leadtime_verify_features)
original_leadtime_pred = leadtime_p95_model.predict(leadtime_verify_features)
if not np.allclose(loaded_leadtime_pred, original_leadtime_pred):
    raise ValueError("저장 후 누적수요 모델 예측이 일치하지 않습니다.")

with open("Bmart_v2_model_config.json", "r", encoding="utf-8") as handle:
    check_config = json.load(handle)

assert check_config["artifact_schema_version"] == "1.1.0"
assert check_config["features"] == feature_columns
assert check_config["target_contract"]["status"] == "warning_unverified_semantics"
assert check_config["monitoring"]["thresholds"] == MONITORING_THRESHOLDS
assert (
    check_config["leadtime_demand_model"]["feature_columns"]
    == leadtime_feature_columns
)
assert check_config["p90_correction"] == P90_CORRECTION
assert check_config["p95_correction"] == P95_CORRECTION
assert (
    check_config["leadtime_demand_model"]["calibration"]["selected_window_days"]
    == LEADTIME_CALIBRATION_WINDOW_DAYS
)

with open("Bmart_v2_artifact_manifest.json", "r", encoding="utf-8") as handle:
    check_manifest = json.load(handle)
for filename, metadata in check_manifest["files"].items():
    current_hash = hashlib.sha256(Path(filename).read_bytes()).hexdigest()
    if current_hash != metadata["sha256"]:
        raise ValueError(f"Artifact hash 불일치: {filename}")

print("모든 Artifact 검증 통과")
print("서비스 CSV shape:", check_service.shape)
print("정적 정책 수:", len(policy_comparison))
print("동적 정책 수:", len(dynamic_policy_comparison))